# NIFTY 200 — BB + Heikin-Ashi Research Scanner (v2)

**Daily live scanner + optional historical research mode.** The three mandatory conditions are
unchanged from v1 and are never silently altered:

```
Condition 1:  Close > Upper Bollinger Band
Condition 2:  0 < BB_Overshoot_Pct <= MAX_BB_OVERSHOOT_PCT   (default 4.0)
Condition 3:  HA_Body_Pct >= MIN_HA_BODY_PCT                 (default 1.0)
```

Everything else — %B, ATR normalization, candle quality, trend/relative-strength/market-regime
context, the breakout state machine, the research/event-study mode, and the ranking score — is
diagnostic, optional, or research-oriented, and is clearly labeled as such.

**What changed vs. v1 (fixing the audit findings):**
- The universe fallback that silently scanned ~20 stocks and called it "NIFTY 200" is **removed**.
  If the official NSE source can't be validated, the notebook **stops** rather than producing a
  partial-universe scan under a NIFTY 200 label.
- The `MIN_REQUIRED_ROWS = 220` bug that rejected valid current constituents (GROWW, ICICIAMC,
  LENSKART, TMCV, etc.) is fixed: the primary scan only needs ~30 sessions; DMA50/DMA200 are now
  *tiered* diagnostics that report `Unavailable — insufficient history` instead of failing the
  whole stock.
- The Excel percentage-format bug (storing `2.31` and formatting as `0.00%`, which Excel renders
  as `231.00%`) is fixed — percentages are now stored as fractions (`0.0231`) with `0.00%` format.

**How to run:** Runtime → Run all. `RUN_RESEARCH_MODE` (Configuration cell) is `False` by default
because the historical event study is a heavier, slower computation — turn it on once the live
scan runs cleanly.

In [1]:
# ============================================================
# SECTION 1 — INSTALL (Colab-only; safe to re-run)
# ============================================================
!pip install -q yfinance==0.2.* openpyxl requests pandas numpy pytz

In [2]:
# ============================================================
# SECTION 2 — IMPORTS & LOGGING
# ============================================================
import os
import io
import time
import pickle
import logging
import warnings
from datetime import datetime, time as dtime, timedelta

import numpy as np
import pandas as pd
import requests
import pytz

try:
    import yfinance as yf
except ImportError:
    raise ImportError("yfinance failed to import — re-run the install cell above.")

from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, PatternFill, Alignment

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("nifty200_research_scanner")

IST = pytz.timezone("Asia/Kolkata")
print("Imports OK.")

Imports OK.


In [3]:
# ============================================================
# SECTION 3 — CONFIGURATION (single source of truth for every parameter)
# ============================================================

# ---------------------------------------------------------------------------
# PART A — MANDATORY PRIMARY STRATEGY (NON-NEGOTIABLE — do not silently change)
# ---------------------------------------------------------------------------
BB_PERIOD            = 20     # Bollinger Bands lookback (trading sessions)
BB_STD_MULT          = 2.0    # Bollinger Bands standard-deviation multiplier
BB_DDOF              = 1      # 1 = sample std (pandas default), 0 = population std. Configurable,
                               # explicit. A different ddof shifts borderline signals slightly —
                               # this is a stated research choice, not hidden inside the code.
MAX_BB_OVERSHOOT_PCT = 4.0    # Max allowed (Close - Upper_BB)/Upper_BB * 100, inclusive
MIN_HA_BODY_PCT      = 1.0    # Min required (HA_Close - HA_Open)/HA_Open * 100

# These remain user-specified strategy parameters, never auto-optimized in this notebook.

# ---------------------------------------------------------------------------
# PART B — DATA PARAMETERS
# ---------------------------------------------------------------------------
HISTORY_PERIOD = "2y"     # ~2 years of daily bars
DATA_INTERVAL  = "1d"     # Daily candles only — no intraday data anywhere in this notebook
AUTO_ADJUST    = True     # See "Corporate action / price basis" note below.

# WHY AUTO_ADJUST=True (documented research choice, not an assumed "correct" answer):
# auto_adjust=True keeps the WHOLE OHLC block (Open/High/Low/Close) on one consistent
# split/dividend-adjusted basis, which is what a rolling-volatility indicator (Bollinger Bands)
# needs to avoid a spurious volatility spike at a corporate-action date. The trade-off, as the
# audit correctly notes, is that adjusted prices can differ from the literal price you would have
# traded at on that date. For a screening tool (not a live order engine) this notebook prioritizes
# internal consistency of the OHLC block. If you need raw/traded prices for actual order placement,
# re-download with AUTO_ADJUST=False in a separate pass — do not mix the two bases in one run.

# ---------------------------------------------------------------------------
# PART C — TIERED MINIMUM HISTORY (fixes the "220 rows rejects valid stocks" bug)
# ---------------------------------------------------------------------------
# The PRIMARY strategy only needs BB_PERIOD (20) sessions plus a small buffer so the last row's
# BB/ATR are not NaN. A stock is rejected ONLY if it cannot support the primary calculation —
# never merely because a diagnostic (DMA200 etc.) is unavailable.
MIN_ROWS_PRIMARY = max(BB_PERIOD, 14) + 10   # = 30 by default: covers BB(20) and ATR(14) + buffer

MIN_ROWS_SMA20   = 20
MIN_ROWS_SMA50   = 50
MIN_ROWS_SMA200  = 200
MIN_ROWS_RS_20D  = 21
MIN_ROWS_RS_60D  = 61

# ---------------------------------------------------------------------------
# PART D — OPTIONAL CONFIRMATION FILTERS (ALL OFF by default; never change the base signal)
# ---------------------------------------------------------------------------
USE_TREND_FILTER              = False   # Close > SMA50
USE_LONG_TREND_FILTER         = False   # Close > SMA200
USE_VOLUME_FILTER             = False   # Volume_Ratio_20 > 1.0
USE_BANDWIDTH_FILTER          = False   # BB width expanding vs. yesterday
USE_ATR_FILTER                = False   # BB_Overshoot_ATR within a "reasonable" band (see below)
USE_CANDLE_QUALITY_FILTER     = False   # Close_Location_Value >= CANDLE_QUALITY_MIN_CLV
USE_RELATIVE_STRENGTH_FILTER  = False   # RS_20D > 0 (stock outperforming the benchmark index)
USE_MARKET_REGIME_FILTER      = False   # benchmark index regime == "Bull Trend"
USE_FRESH_BREAKOUT_ONLY       = False   # Breakout_Type == "Fresh Breakout"

# Thresholds used only when the corresponding optional filter above is enabled:
ATR_OVERSHOOT_MAX      = 1.5   # BB_Overshoot_ATR <= this is treated as "not an outlier-sized move"
CANDLE_QUALITY_MIN_CLV = 0.5   # Close_Location_Value >= this = close nearer the day's high

# ---------------------------------------------------------------------------
# PART E — UNIVERSE SOURCE & INTEGRITY (hard-fail policy — see audit point 2)
# ---------------------------------------------------------------------------
NSE_NIFTY200_URLS = [
    "https://nsearchives.nseindia.com/content/indices/ind_nifty200list.csv",
    "https://archives.nseindia.com/content/indices/ind_nifty200list.csv",
    "https://www1.nseindia.com/content/indices/ind_nifty200list.csv",
]
YFINANCE_SUFFIX = ".NS"

# NIFTY 200 = NIFTY 100 + NIFTY MIDCAP 100 by NSE's own definition, so ~200 is expected but
# minor drift around index rebalances is normal. Anything far outside this band is untrustworthy.
UNIVERSE_MIN_COUNT = 150
UNIVERSE_MAX_COUNT = 210

# If ALL official NSE URLs fail OR the file fails structural validation, this notebook STOPS
# (raises) instead of silently substituting a small hard-coded list — UNLESS you provide your own
# previously-downloaded constituent CSV here (columns needed: Symbol [+ optional Company Name /
# Industry]). Leave as None to require the live official source.
CONSTITUENT_OVERRIDE_CSV_PATH = None   # e.g. "/content/ind_nifty200list.csv"

# ---------------------------------------------------------------------------
# PART F — BENCHMARK INDEX (for Relative Strength + Market Regime)
# ---------------------------------------------------------------------------
# yfinance does not reliably expose a dedicated "NIFTY 200 index" ticker; NIFTY 50 (^NSEI) is the
# most robustly available broad NSE benchmark on Yahoo Finance, so it is used as the default
# benchmark for both Relative Strength and Market Regime. This is a documented substitution, not
# a silent one — swap INDEX_TICKER if you have a better-covered NIFTY 200 index source.
INDEX_TICKER = "^NSEI"
INDEX_NAME = "NIFTY 50"

# ---------------------------------------------------------------------------
# PART G — CACHING (avoid re-downloading everything on every re-run)
# ---------------------------------------------------------------------------
CACHE_DIR = "/content/nifty200_cache/"
CACHE_ENABLED = True
FORCE_REDOWNLOAD = False   # set True to bypass cache entirely for this run

# ---------------------------------------------------------------------------
# PART H — BATCH DOWNLOAD / RETRY SETTINGS
# ---------------------------------------------------------------------------
BATCH_SIZE = 50
BATCH_PAUSE_SEC = 1.0
MAX_DOWNLOAD_RETRIES = 2
RETRY_BACKOFF_SEC = 3.0

# ---------------------------------------------------------------------------
# PART I — RESEARCH / HISTORICAL EVENT STUDY (OFF by default — heavier computation)
# ---------------------------------------------------------------------------
RUN_RESEARCH_MODE = False

# Entry timing (choose ONE, document it, never mix): signal detected on completed close of day T.
#   "next_open"  -> entry price = Open of day T+1  (default; avoids same-day-close lookahead)
#   "next_close" -> entry price = Close of day T+1
ENTRY_PRICE_METHOD = "next_open"

FORWARD_RETURN_HORIZONS = [1, 3, 5, 10, 20]   # trading days held after entry
PRIMARY_RESEARCH_HORIZON = 5 if 5 in FORWARD_RETURN_HORIZONS else FORWARD_RETURN_HORIZONS[0]

# --- Survivorship-bias disclosure (see audit point 19 / Part 20) ---
# This notebook only has TODAY's NIFTY 200 constituent list, not historical membership as of each
# past signal date. Any historical research below is therefore explicitly labeled a
# "CURRENT-UNIVERSE HISTORICAL SIMULATION" and is NOT an unbiased point-in-time NIFTY 200 backtest.
RESEARCH_MODE_LABEL = "CURRENT-UNIVERSE HISTORICAL SIMULATION (survivorship bias present)"

# --- Walk-forward split (descriptive only — NOT used to auto-optimize anything) ---
TRAIN_START = "2023-01-01"
TRAIN_END   = "2024-06-30"
TEST_START  = "2024-07-01"
TEST_END    = "2100-01-01"   # effectively "through the latest available data"

# --- Transaction cost assumptions (applied only to NET forward returns in research mode) ---
BROKERAGE_PCT        = 0.03    # per side, % of trade value (adjust to your broker)
STT_PCT              = 0.10    # Securities Transaction Tax, sell side, % (illustrative)
EXCHANGE_CHARGES_PCT = 0.0035
GST_PCT              = 0.18    # applied on brokerage + exchange charges, not on trade value
SLIPPAGE_PCT         = 0.05    # per side, % of trade value

def _round_trip_cost_pct():
    """Rough round-trip cost estimate in % of trade value. Illustrative, not a tax/brokerage
    quote — adjust the components above to your actual broker and instrument."""
    per_side_charges = BROKERAGE_PCT + EXCHANGE_CHARGES_PCT + SLIPPAGE_PCT
    gst_on_charges = (BROKERAGE_PCT + EXCHANGE_CHARGES_PCT) * GST_PCT
    return 2 * per_side_charges + gst_on_charges + STT_PCT

ROUND_TRIP_COST_PCT = _round_trip_cost_pct()

# ---------------------------------------------------------------------------
# PART J — SENSITIVITY GRIDS (descriptive only — never auto-select a "best" combo)
# ---------------------------------------------------------------------------
SENS_BB_PERIOD_GRID   = [15, 20, 25]
SENS_BB_STD_MULT_GRID = [1.5, 2.0, 2.5]
SENS_OVERSHOOT_GRID   = [2.0, 3.0, 4.0, 5.0]
SENS_HA_BODY_GRID     = [0.5, 1.0, 1.5, 2.0]

# ---------------------------------------------------------------------------
# PART K — SIGNAL QUALITY SCORE WEIGHTS (documented heuristic — see Ranking section)
# ---------------------------------------------------------------------------
SCORE_WEIGHTS = {
    "Trend_Score": 0.25,
    "Momentum_Score": 0.20,
    "Volatility_Score": 0.15,
    "Candle_Quality_Score": 0.15,
    "Liquidity_Score": 0.10,
    "Market_Regime_Score": 0.15,
}
assert abs(sum(SCORE_WEIGHTS.values()) - 1.0) < 1e-9, "SCORE_WEIGHTS must sum to 1.0"

# ---------------------------------------------------------------------------
# PART L — OUTPUT
# ---------------------------------------------------------------------------
OUTPUT_FILENAME = "Nifty200_BB_HA_Research_Scanner.xlsx"

RUN_TIMESTAMP_IST = datetime.now(IST)
os.makedirs(CACHE_DIR, exist_ok=True)

print("Configuration loaded.")
print(f"  MANDATORY -> BB_PERIOD={BB_PERIOD} BB_STD_MULT={BB_STD_MULT} BB_DDOF={BB_DDOF} "
      f"MAX_BB_OVERSHOOT_PCT={MAX_BB_OVERSHOOT_PCT} MIN_HA_BODY_PCT={MIN_HA_BODY_PCT}")
print(f"  History tiers -> primary>={MIN_ROWS_PRIMARY} sma50>={MIN_ROWS_SMA50} sma200>={MIN_ROWS_SMA200}")
print(f"  Universe integrity -> accept range [{UNIVERSE_MIN_COUNT}, {UNIVERSE_MAX_COUNT}], "
      f"override_csv={CONSTITUENT_OVERRIDE_CSV_PATH}")
print(f"  Benchmark index -> {INDEX_TICKER} ({INDEX_NAME})")
print(f"  RUN_RESEARCH_MODE={RUN_RESEARCH_MODE}  ROUND_TRIP_COST_PCT~={ROUND_TRIP_COST_PCT:.3f}%")
print(f"  Run timestamp (IST): {RUN_TIMESTAMP_IST.strftime('%Y-%m-%d %H:%M:%S %Z')}")

Configuration loaded.
  MANDATORY -> BB_PERIOD=20 BB_STD_MULT=2.0 BB_DDOF=1 MAX_BB_OVERSHOOT_PCT=4.0 MIN_HA_BODY_PCT=1.0
  History tiers -> primary>=30 sma50>=50 sma200>=200
  Universe integrity -> accept range [150, 210], override_csv=None
  Benchmark index -> ^NSEI (NIFTY 50)
  RUN_RESEARCH_MODE=False  ROUND_TRIP_COST_PCT~=0.273%
  Run timestamp (IST): 2026-09-09 08:55:29 IST


## Universe acquisition — hard-fail policy (audit fix)

**This replaces the v1 behavior.** v1 fell back to a ~20-stock emergency list if every official NSE
URL failed, and the notebook could then finish and print "scan complete" while having actually
scanned a tiny partial universe under the NIFTY 200 label. That is exactly the failure mode the
audit flagged as unsafe.

v2 policy: try the official NSE URLs; if none validate, **raise and stop** — unless you've set
`CONSTITUENT_OVERRIDE_CSV_PATH` to a constituent file you trust. There is no silent partial-universe
fallback.

In [4]:
# ============================================================
# SECTION 4a — NIFTY 200 UNIVERSE ACQUISITION (hard-fail, no silent partial-universe fallback)
# ============================================================
NSE_HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                    "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"),
    "Accept": "text/csv,application/csv,text/plain,*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.niftyindices.com/",
}


class UniverseIntegrityError(RuntimeError):
    """Raised when the NIFTY 200 constituent universe cannot be validated. This is intentionally
    NOT caught anywhere upstream — a failed universe must stop the notebook, not degrade into a
    partial scan silently mislabeled as NIFTY 200."""
    pass


def _parse_constituent_csv(text: str, source_label: str) -> pd.DataFrame:
    df = pd.read_csv(io.StringIO(text))
    df.columns = [c.strip() for c in df.columns]

    col_map = {}
    for c in df.columns:
        lc = c.lower()
        if "symbol" in lc:
            col_map[c] = "Symbol"
        elif "company" in lc:
            col_map[c] = "Company_Name"
        elif "industry" in lc or "sector" in lc:
            col_map[c] = "Sector"
    df = df.rename(columns=col_map)

    if "Symbol" not in df.columns:
        raise UniverseIntegrityError(
            f"[{source_label}] 'Symbol' column not found. Columns present: {list(df.columns)}"
        )

    df["Symbol"] = df["Symbol"].astype(str).str.strip().str.upper()
    df = df[df["Symbol"].str.len() > 0].reset_index(drop=True)

    n_raw = len(df)
    n_dupes = int(df["Symbol"].duplicated().sum())
    if n_dupes > 0:
        logger.warning(f"[{source_label}] {n_dupes} duplicate symbols found — de-duplicating.")
        df = df.drop_duplicates(subset="Symbol").reset_index(drop=True)

    if not (UNIVERSE_MIN_COUNT <= len(df) <= UNIVERSE_MAX_COUNT):
        raise UniverseIntegrityError(
            f"[{source_label}] constituent count {len(df)} (raw {n_raw}, {n_dupes} dupes removed) "
            f"is outside the accepted NIFTY 200 range [{UNIVERSE_MIN_COUNT}, {UNIVERSE_MAX_COUNT}]. "
            f"Refusing to treat this as a valid NIFTY 200 universe."
        )

    if "Company_Name" not in df.columns:
        df["Company_Name"] = df["Symbol"]
    if "Sector" not in df.columns:
        df["Sector"] = np.nan

    return df[["Symbol", "Company_Name", "Sector"]]


def get_nifty200_constituents(urls=NSE_NIFTY200_URLS, override_csv_path=CONSTITUENT_OVERRIDE_CSV_PATH,
                               timeout=15):
    """
    Returns (constituents_df, source_used, retrieval_timestamp, note).
    Raises UniverseIntegrityError if no source can be validated — this is intentional and must
    not be caught by a broad try/except upstream (see Section 26 error-handling stages: 'universe'
    failures are fatal, not per-ticker).
    """
    retrieval_ts = datetime.now(IST)

    if override_csv_path:
        if not os.path.exists(override_csv_path):
            raise UniverseIntegrityError(
                f"CONSTITUENT_OVERRIDE_CSV_PATH is set to '{override_csv_path}' but that file "
                f"does not exist. Fix the path or set it back to None to use the live official source."
            )
        with open(override_csv_path, "r", encoding="utf-8") as f:
            text = f.read()
        df = _parse_constituent_csv(text, f"USER_OVERRIDE:{override_csv_path}")
        note = "User-supplied override CSV — validate that this reflects the intended date's membership."
        logger.info(f"Universe source: user override CSV ({len(df)} constituents).")
        return df, f"USER_OVERRIDE_CSV:{override_csv_path}", retrieval_ts, note

    session = requests.Session()
    session.headers.update(NSE_HEADERS)
    try:
        session.get("https://www.nseindia.com", timeout=timeout)
    except Exception as e:
        logger.warning(f"NSE homepage warm-up request failed (continuing anyway): {e}")

    failures = []
    for url in urls:
        try:
            resp = session.get(url, timeout=timeout)
            resp.raise_for_status()
            df = _parse_constituent_csv(resp.text, url)
            note = f"Official NSE indices file, {len(df)} constituents, validated in [{UNIVERSE_MIN_COUNT}, {UNIVERSE_MAX_COUNT}]."
            logger.info(f"Universe source OK: {url} ({len(df)} constituents)")
            return df, url, retrieval_ts, note
        except Exception as e:
            failures.append(f"{url} -> {type(e).__name__}: {e}")
            logger.warning(f"Universe source failed [{url}]: {type(e).__name__}: {e}")
            continue

    # --- HARD FAIL: no silent fallback to a small stale list ---
    failure_report = "\n".join(f"  - {f}" for f in failures)
    raise UniverseIntegrityError(
        "Could not retrieve a valid NIFTY 200 constituent list from ANY official NSE source, and "
        "no CONSTITUENT_OVERRIDE_CSV_PATH was supplied. Per design, this notebook does NOT fall "
        "back to a small hard-coded list and call it a NIFTY 200 scan.\n"
        f"Attempts:\n{failure_report}\n\n"
        "To proceed: (a) re-run later if this is a transient NSE outage, or (b) manually download "
        "the current constituent CSV (e.g. from niftyindices.com), upload it to this Colab session, "
        "and set CONSTITUENT_OVERRIDE_CSV_PATH to its path in the Configuration cell, then re-run."
    )


constituents_df, universe_source, universe_retrieval_ts, universe_note = get_nifty200_constituents()
print(f"Universe source     : {universe_source}")
print(f"Retrieval timestamp : {universe_retrieval_ts.strftime('%Y-%m-%d %H:%M:%S %Z')}")
print(f"Universe note       : {universe_note}")
print(f"Constituents found  : {len(constituents_df)}")
constituents_df.head()

Universe source     : https://nsearchives.nseindia.com/content/indices/ind_nifty200list.csv
Retrieval timestamp : 2026-09-09 08:55:29 IST
Universe note       : Official NSE indices file, 200 constituents, validated in [150, 210].
Constituents found  : 200


,Symbol,Company_Name,Sector
0,360ONE,360 ONE WAM Ltd.,Financial Services
1,ABB,ABB India Ltd.,Capital Goods
2,APLAPOLLO,APL Apollo Tubes Ltd.,Capital Goods
3,AUBANK,AU Small Finance Bank Ltd.,Financial Services
4,ADANIENSOL,Adani Energy Solutions Ltd.,Power


In [5]:
# ============================================================
# SECTION 4b — SYSTEMATIC SYMBOL MAPPING (NSE -> yfinance), fully auditable
# ============================================================
# A small, explicit override table for symbols where the yfinance ticker does not match a plain
# "<NSE_SYMBOL>.NS" pattern. Extend this table as mapping failures are discovered (see
# Mapping_Status in the Universe sheet after a run) rather than guessing silently.
NSE_TO_YFINANCE_OVERRIDES = {
    "M&M": "M&M",
    "M&MFIN": "M&MFIN",
    "L&TFH": "L&TFH",
    "BAJAJ-AUTO": "BAJAJ-AUTO",
}


def map_symbol_to_yfinance(nse_symbol: str):
    """Returns (yfinance_symbol, mapping_method)."""
    if nse_symbol in NSE_TO_YFINANCE_OVERRIDES:
        base = NSE_TO_YFINANCE_OVERRIDES[nse_symbol]
        return f"{base}{YFINANCE_SUFFIX}", "override_table"
    return f"{nse_symbol}{YFINANCE_SUFFIX}", "direct_suffix"


_mapped = constituents_df["Symbol"].apply(map_symbol_to_yfinance)
constituents_df["YFinance_Symbol"] = _mapped.apply(lambda t: t[0])
constituents_df["Mapping_Method"] = _mapped.apply(lambda t: t[1])
constituents_df["Mapping_Status"] = "PENDING"   # finalized once download/validation results are known

print("Symbol mapping methods used:")
print(constituents_df["Mapping_Method"].value_counts())
constituents_df[["Symbol", "YFinance_Symbol", "Mapping_Method"]].head(10)

Symbol mapping methods used:
Mapping_Method
direct_suffix     197
override_table      3
Name: count, dtype: int64


,Symbol,YFinance_Symbol,Mapping_Method
0,360ONE,360ONE.NS,direct_suffix
1,ABB,ABB.NS,direct_suffix
2,APLAPOLLO,APLAPOLLO.NS,direct_suffix
3,AUBANK,AUBANK.NS,direct_suffix
4,ADANIENSOL,ADANIENSOL.NS,direct_suffix
5,ADANIENT,ADANIENT.NS,direct_suffix
6,ADANIGREEN,ADANIGREEN.NS,direct_suffix
7,ADANIPORTS,ADANIPORTS.NS,direct_suffix
8,ADANIPOWER,ADANIPOWER.NS,direct_suffix
9,ATGL,ATGL.NS,direct_suffix


## Data download — batched, retried, cached

Downloads are batched (kinder to the provider than one request per ticker), retried on failure
with backoff, and cached to disk for the day so a re-run of the same session doesn't re-download
everything. `auto_adjust` is passed explicitly (see Configuration).

In [6]:
# ============================================================
# SECTION 5a — CACHE HELPERS
# ============================================================
def _cache_path(yf_symbol: str, period: str, interval: str, auto_adjust: bool) -> str:
    safe = yf_symbol.replace(".", "_").replace("&", "AND")
    key = f"{safe}__{period}__{interval}__adj{int(auto_adjust)}.pkl"
    return os.path.join(CACHE_DIR, key)


def load_from_cache(yf_symbol: str, period=HISTORY_PERIOD, interval=DATA_INTERVAL,
                     auto_adjust=AUTO_ADJUST, max_age_hours=20):
    if not CACHE_ENABLED or FORCE_REDOWNLOAD:
        return None
    path = _cache_path(yf_symbol, period, interval, auto_adjust)
    if not os.path.exists(path):
        return None
    try:
        with open(path, "rb") as f:
            payload = pickle.load(f)
        cached_at = payload.get("cached_at")
        if cached_at is None:
            return None
        age_hours = (datetime.now(IST) - cached_at).total_seconds() / 3600.0
        if age_hours > max_age_hours:
            return None
        return payload.get("df")
    except Exception as e:
        logger.warning(f"Cache read failed for {yf_symbol}: {type(e).__name__}: {e}")
        return None


def save_to_cache(yf_symbol: str, df: pd.DataFrame, period=HISTORY_PERIOD, interval=DATA_INTERVAL,
                   auto_adjust=AUTO_ADJUST):
    if not CACHE_ENABLED or df is None:
        return
    path = _cache_path(yf_symbol, period, interval, auto_adjust)
    try:
        with open(path, "wb") as f:
            pickle.dump({"df": df, "cached_at": datetime.now(IST)}, f)
    except Exception as e:
        logger.warning(f"Cache write failed for {yf_symbol}: {type(e).__name__}: {e}")

In [7]:
# ============================================================
# SECTION 5b — BATCH OHLC DOWNLOAD (with cache + retries)
# ============================================================
def _download_batch_once(chunk, period, interval, auto_adjust):
    return yf.download(
        tickers=chunk,
        period=period,
        interval=interval,
        auto_adjust=auto_adjust,
        group_by="ticker",
        threads=True,
        progress=False,
    )


def download_ohlc_batch(symbols, period=HISTORY_PERIOD, interval=DATA_INTERVAL,
                         auto_adjust=AUTO_ADJUST, batch_size=BATCH_SIZE, pause=BATCH_PAUSE_SEC,
                         max_retries=MAX_DOWNLOAD_RETRIES, backoff=RETRY_BACKOFF_SEC):
    """
    Returns (results: dict[symbol -> DataFrame|None], download_errors: dict[symbol -> str],
             cache_hits: set[symbol]).
    """
    results = {}
    download_errors = {}
    cache_hits = set()
    to_download = []

    for sym in symbols:
        cached = load_from_cache(sym, period, interval, auto_adjust)
        if cached is not None:
            results[sym] = cached
            cache_hits.add(sym)
        else:
            to_download.append(sym)

    if cache_hits:
        print(f"Loaded {len(cache_hits)}/{len(symbols)} symbols from cache "
              f"(cache dir: {CACHE_DIR}, disable with CACHE_ENABLED=False).")

    n_batches = (len(to_download) + batch_size - 1) // batch_size if to_download else 0
    for b in range(n_batches):
        chunk = to_download[b * batch_size:(b + 1) * batch_size]
        print(f"Downloading batch {b + 1}/{n_batches} ({len(chunk)} symbols)...")

        data = None
        last_err = None
        for attempt in range(max_retries + 1):
            try:
                data = _download_batch_once(chunk, period, interval, auto_adjust)
                break
            except Exception as e:
                last_err = e
                if attempt < max_retries:
                    logger.warning(f"Batch {b + 1} attempt {attempt + 1} failed "
                                    f"({type(e).__name__}: {e}); retrying in {backoff}s...")
                    time.sleep(backoff)
                else:
                    logger.error(f"Batch {b + 1} failed after {max_retries + 1} attempts: {e}")

        if data is None:
            msg = f"batch_download_failed_after_retries: {type(last_err).__name__}: {last_err}"
            for sym in chunk:
                results[sym] = None
                download_errors[sym] = msg
            time.sleep(pause)
            continue

        for sym in chunk:
            try:
                if len(chunk) == 1:
                    df_sym = data.copy()
                elif isinstance(data.columns, pd.MultiIndex) and sym in data.columns.get_level_values(0):
                    df_sym = data[sym].copy()
                else:
                    df_sym = None

                if df_sym is None or df_sym.dropna(how="all").empty:
                    results[sym] = None
                    download_errors[sym] = "no_data_returned"
                else:
                    results[sym] = df_sym
                    save_to_cache(sym, df_sym, period, interval, auto_adjust)
            except Exception as e:
                results[sym] = None
                download_errors[sym] = f"{type(e).__name__}: {e}"

        time.sleep(pause)

    n_ok = sum(1 for v in results.values() if v is not None)
    n_fresh = n_ok - len(cache_hits)
    print(f"\nDownload complete: {n_ok}/{len(results)} symbols have data "
          f"({len(cache_hits)} from cache, {n_fresh} freshly downloaded).")
    return results, download_errors, cache_hits


raw_ohlc_data, download_error_log, cache_hit_symbols = download_ohlc_batch(
    constituents_df["YFinance_Symbol"].tolist()
)

# --- Benchmark index download (for Relative Strength + Market Regime) ---
print(f"\nDownloading benchmark index {INDEX_TICKER} ({INDEX_NAME})...")
_index_raw, _index_err, _ = download_ohlc_batch([INDEX_TICKER])
index_raw_df = _index_raw.get(INDEX_TICKER)
if index_raw_df is None:
    logger.warning(f"Benchmark index download failed: {_index_err.get(INDEX_TICKER)}. "
                    f"Relative Strength and Market Regime will be unavailable this run.")


Download complete: 200/200 symbols have data (0 from cache, 200 freshly downloaded).


Download complete: 1/1 symbols have data (0 from cache, 1 freshly downloaded).


## Data quality validation — tiered (audit fix)

**This is the fix for the "220 rows rejects valid stocks" bug.** A stock is only rejected if it
cannot support the PRIMARY calculation (`MIN_ROWS_PRIMARY`, default 30 sessions). Diagnostics that
need more history (SMA50, SMA200, RS_60D, ...) are computed when available and explicitly marked
`Unavailable — insufficient history` otherwise — they never fail the whole stock.

In [8]:
# ============================================================
# SECTION 6 — DATA QUALITY VALIDATION (tiered) + INCOMPLETE-CANDLE / STALENESS CHECK
# ============================================================
REQUIRED_COLS = ["Open", "High", "Low", "Close", "Volume"]
NSE_CLOSE_TIME = dtime(15, 30)
STALE_DATA_MAX_DAYS = 5   # if the newest bar is older than this many calendar days, flag Data_Stale


def validate_and_clean_ohlc(df: pd.DataFrame, min_rows=MIN_ROWS_PRIMARY):
    """
    Validates and cleans a single stock's OHLCV DataFrame. Rejects ONLY if the PRIMARY calculation
    cannot be supported (fewer than `min_rows` valid rows) — this is deliberately much looser than
    v1's flat 220-row requirement, which incorrectly rejected valid, recently-listed constituents.
    Raises ValueError with a clear message on any failure (caught by the caller).
    """
    if df is None or df.empty:
        raise ValueError("no data returned")

    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"missing required columns: {missing_cols}")

    df = df[REQUIRED_COLS].copy()
    for c in REQUIRED_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df[~df.index.duplicated(keep="last")]
    df = df.sort_index()

    df["Volume"] = df["Volume"].fillna(0)
    df = df.dropna(subset=["Open", "High", "Low", "Close"])
    df = df[(df["Open"] > 0) & (df["High"] > 0) & (df["Low"] > 0) & (df["Close"] > 0)]

    if len(df) < min_rows:
        raise ValueError(
            f"insufficient history for PRIMARY calculation: {len(df)} valid rows "
            f"(need >= {min_rows}). This stock is skipped, not silently included with NaN signals."
        )

    return df


def trim_incomplete_candle(df: pd.DataFrame, now_ist: datetime):
    """
    Drops the last row if it is *today's* (IST) bar and the NSE regular session has not yet
    closed. Returns (trimmed_df, was_trimmed: bool).
    """
    last_ts = df.index[-1]
    last_date = last_ts.date() if hasattr(last_ts, "date") else pd.Timestamp(last_ts).date()

    if last_date == now_ist.date() and now_ist.time() < NSE_CLOSE_TIME:
        return df.iloc[:-1].copy(), True
    return df, False


def assess_staleness(df: pd.DataFrame, now_ist: datetime):
    """
    Flags Data_Stale=True if the newest available bar is older than STALE_DATA_MAX_DAYS calendar
    days relative to 'now'. This does NOT reject the stock — it is reported so stale data is never
    silently treated as current (audit / Part 5 requirement).
    """
    last_ts = df.index[-1]
    last_date = last_ts.date() if hasattr(last_ts, "date") else pd.Timestamp(last_ts).date()
    age_days = (now_ist.date() - last_date).days
    return bool(age_days > STALE_DATA_MAX_DAYS), age_days

## Indicator engine

Bollinger Bands on regular Close only, with a configurable `ddof` and the addition of `%B`
(Bollinger's own normalized breakout measure). Heikin-Ashi is recursive from raw OHLC, extended
with wick diagnostics. ATR(14) is added for volatility normalization of the overshoot.

In [9]:
# ============================================================
# SECTION 7a — BOLLINGER ENGINE (regular Close only; configurable ddof; adds %B)
# ============================================================
def calculate_bollinger_bands(df: pd.DataFrame, period=BB_PERIOD, std_mult=BB_STD_MULT, ddof=BB_DDOF) -> pd.DataFrame:
    """
    Middle = SMA(Close, period)
    StdDev = rolling standard deviation of Close over `period`, explicit ddof (0=population, 1=sample)
    Upper  = Middle + std_mult * StdDev
    Lower  = Middle - std_mult * StdDev
    Width% = (Upper - Lower) / Middle * 100
    %B     = (Close - Lower) / (Upper - Lower)   -- Bollinger's own normalized location measure.
             %B < 0 : below lower band | 0-1 : inside bands | %B > 1 : above upper band.
             This is an ADDITIONAL diagnostic, never a replacement for the mandatory rule.
    """
    bb = pd.DataFrame(index=df.index)
    bb["BB_Middle"] = df["Close"].rolling(window=period, min_periods=period).mean()
    bb["BB_StdDev"] = df["Close"].rolling(window=period, min_periods=period).std(ddof=ddof)
    bb["BB_Upper"] = bb["BB_Middle"] + std_mult * bb["BB_StdDev"]
    bb["BB_Lower"] = bb["BB_Middle"] - std_mult * bb["BB_StdDev"]
    bb["BB_Width_Pct"] = (bb["BB_Upper"] - bb["BB_Lower"]) / bb["BB_Middle"] * 100.0

    band_range = bb["BB_Upper"] - bb["BB_Lower"]
    bb["BB_PctB"] = np.where(band_range > 0, (df["Close"] - bb["BB_Lower"]) / band_range, np.nan)
    return bb

In [10]:
# ============================================================
# SECTION 7b — HEIKIN-ASHI ENGINE (recursive; extended with wick diagnostics)
# ============================================================
def calculate_heikin_ashi(df: pd.DataFrame) -> pd.DataFrame:
    """
    HA_Close[t] = (Open[t]+High[t]+Low[t]+Close[t])/4
    HA_Open[0]  = (Open[0]+Close[0])/2                       (seed)
    HA_Open[t]  = (HA_Open[t-1]+HA_Close[t-1])/2              for t>0   (explicit loop: this
                  recursion has no correct vectorized closed form)
    HA_High[t]  = max(High[t], HA_Open[t], HA_Close[t])
    HA_Low[t]   = min(Low[t],  HA_Open[t], HA_Close[t])

    IMPORTANT: Heikin-Ashi is a synthetic/averaged candle. It is used here ONLY to measure candle
    "conviction" (body strength) for the mandatory condition — never as an actual execution price,
    and never fed back into the Bollinger calculation.
    """
    ha_close = (df["Open"] + df["High"] + df["Low"] + df["Close"]) / 4.0
    ha_close_vals = ha_close.values

    ha_open_vals = np.empty(len(df))
    ha_open_vals[0] = (df["Open"].iloc[0] + df["Close"].iloc[0]) / 2.0
    for i in range(1, len(df)):
        ha_open_vals[i] = (ha_open_vals[i - 1] + ha_close_vals[i - 1]) / 2.0

    ha = pd.DataFrame(index=df.index)
    ha["HA_Close"] = ha_close
    ha["HA_Open"] = ha_open_vals
    ha["HA_High"] = pd.concat([df["High"], ha["HA_Open"], ha["HA_Close"]], axis=1).max(axis=1)
    ha["HA_Low"] = pd.concat([df["Low"], ha["HA_Open"], ha["HA_Close"]], axis=1).min(axis=1)

    ha_upper_body = pd.concat([ha["HA_Open"], ha["HA_Close"]], axis=1).max(axis=1)
    ha_lower_body = pd.concat([ha["HA_Open"], ha["HA_Close"]], axis=1).min(axis=1)
    ha_range = ha["HA_High"] - ha["HA_Low"]

    ha["HA_Upper_Wick_Pct"] = (ha["HA_High"] - ha_upper_body) / ha["HA_Open"] * 100.0
    ha["HA_Lower_Wick_Pct"] = (ha_lower_body - ha["HA_Low"]) / ha["HA_Open"] * 100.0
    ha["HA_Body_to_Range"] = np.where(ha_range > 0, (ha["HA_Close"] - ha["HA_Open"]).abs() / ha_range, np.nan)

    return ha

In [11]:
# ============================================================
# SECTION 7c — ATR ENGINE (volatility normalization)
# ============================================================
def calculate_atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    """
    Wilder's ATR(period) using True Range = max(High-Low, |High-PrevClose|, |Low-PrevClose|),
    smoothed with a simple rolling mean (a common, transparent approximation of Wilder smoothing;
    documented here rather than silently assumed).
    """
    prev_close = df["Close"].shift(1)
    tr = pd.concat([
        df["High"] - df["Low"],
        (df["High"] - prev_close).abs(),
        (df["Low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr.rolling(window=period, min_periods=period).mean()

## Feature engine — candle quality, volume/liquidity, trend structure

Trend diagnostics (SMA50/SMA200 etc.) use the **tiered availability** pattern: computed when there
is enough history, otherwise `NaN` with an explicit `*_Status = "Unavailable — insufficient
history"` — never used to reject the stock.

In [12]:
# ============================================================
# SECTION 8a — CANDLE QUALITY (regular candle, not Heikin-Ashi)
# ============================================================
def calculate_candle_quality(df: pd.DataFrame) -> pd.DataFrame:
    """
    Candle_Body_Pct, wick sizes, and Close_Location_Value (CLV) on the REGULAR candle. CLV
    distinguishes a strong close near the high from a breakout attempt with a large rejection wick:
        CLV = ((Close - Low) - (High - Close)) / (High - Low)   in [-1, 1], guarded for High==Low.
    """
    cq = pd.DataFrame(index=df.index)
    o, h, l, c = df["Open"], df["High"], df["Low"], df["Close"]
    rng = h - l

    cq["Candle_Body_Pct"] = (c - o) / o * 100.0
    cq["Upper_Wick_Pct"] = (h - pd.concat([o, c], axis=1).max(axis=1)) / o * 100.0
    cq["Lower_Wick_Pct"] = (pd.concat([o, c], axis=1).min(axis=1) - l) / o * 100.0
    cq["Range_Pct"] = rng / o * 100.0
    cq["Close_Location_Value"] = np.where(rng > 0, ((c - l) - (h - c)) / rng, np.nan)
    return cq

In [13]:
# ============================================================
# SECTION 8b — VOLUME / LIQUIDITY
# ============================================================
def calculate_volume_liquidity(df: pd.DataFrame) -> pd.DataFrame:
    """
    NOTE: more volume is NOT assumed to be automatically bullish (see audit point 11) — these are
    diagnostic fields only, feeding an OPTIONAL filter and the ranking score, never the mandatory
    signal.
    """
    vl = pd.DataFrame(index=df.index)
    vl["Avg_Volume_20"] = df["Volume"].rolling(window=20, min_periods=20).mean()
    vl["Avg_Volume_50"] = df["Volume"].rolling(window=50, min_periods=50).mean()
    vl["Volume_Ratio_20"] = df["Volume"] / vl["Avg_Volume_20"]
    vl["Volume_Ratio_50"] = df["Volume"] / vl["Avg_Volume_50"]
    vl["Dollar_Volume"] = df["Volume"] * df["Close"]
    return vl

In [14]:
# ============================================================
# SECTION 8c — TREND STRUCTURE (tiered availability — fixes the "reject on short history" bug)
# ============================================================
def _tiered_sma(close: pd.Series, window: int, min_rows_needed: int):
    """Returns (sma_series, status_string). status is computed once (based on total available
    rows), not per-row, to keep it simple and auditable in the final output row."""
    sma = close.rolling(window=window, min_periods=window).mean()
    status = "OK" if len(close) >= min_rows_needed else "Unavailable — insufficient history"
    return sma, status


def calculate_trend_structure(df: pd.DataFrame) -> pd.DataFrame:
    ts = pd.DataFrame(index=df.index)
    close = df["Close"]

    ts["SMA20"], sma20_status = _tiered_sma(close, 20, MIN_ROWS_SMA20)
    ts["SMA50"], sma50_status = _tiered_sma(close, 50, MIN_ROWS_SMA50)
    ts["SMA200"], sma200_status = _tiered_sma(close, 200, MIN_ROWS_SMA200)

    ts["SMA50_Status"] = sma50_status
    ts["SMA200_Status"] = sma200_status

    ts["Dist_SMA50_Pct"] = (close - ts["SMA50"]) / ts["SMA50"] * 100.0
    ts["Dist_SMA200_Pct"] = (close - ts["SMA200"]) / ts["SMA200"] * 100.0

    # Slopes: % change of the moving average itself over a lookback window (simple, transparent).
    ts["SMA20_Slope_10D"] = (ts["SMA20"] - ts["SMA20"].shift(10)) / ts["SMA20"].shift(10) * 100.0
    ts["SMA50_Slope_20D"] = (ts["SMA50"] - ts["SMA50"].shift(20)) / ts["SMA50"].shift(20) * 100.0
    ts["SMA200_Slope_20D"] = (ts["SMA200"] - ts["SMA200"].shift(20)) / ts["SMA200"].shift(20) * 100.0

    return ts

## Market regime & relative strength

Both are computed from the benchmark index (`INDEX_TICKER`, default NIFTY 50 — see Configuration
note on why NIFTY 200's own index isn't used). Regime classification uses an explicit, parameterized
rule (not an optimized threshold), and stays optional (`USE_MARKET_REGIME_FILTER=False` by default).

In [15]:
# ============================================================
# SECTION 8d — MARKET REGIME (benchmark index; optional filter; parameterized, not optimized)
# ============================================================
def calculate_index_features(index_df_clean: pd.DataFrame) -> pd.DataFrame:
    """
    Builds a date-indexed DataFrame of benchmark-index diagnostics used for both Market Regime
    and Relative Strength. Regime rule (explicit, not tuned):
        Bull Trend : Index_Close > Index_SMA200 AND Index_SMA50 > Index_SMA200
        Bear Trend : Index_Close < Index_SMA200 AND Index_SMA50 < Index_SMA200
        Neutral    : anything else
        Unavailable — insufficient index history : if SMA200 cannot be computed yet
    """
    idx = pd.DataFrame(index=index_df_clean.index)
    close = index_df_clean["Close"]
    idx["Index_Close"] = close
    idx["Index_SMA20"], _ = _tiered_sma(close, 20, MIN_ROWS_SMA20)
    idx["Index_SMA50"], _ = _tiered_sma(close, 50, MIN_ROWS_SMA50)
    idx["Index_SMA200"], _ = _tiered_sma(close, 200, MIN_ROWS_SMA200)
    idx["Index_Return_20D"] = (close / close.shift(20) - 1) * 100.0
    idx["Index_Return_60D"] = (close / close.shift(60) - 1) * 100.0

    def _classify(row):
        if pd.isna(row["Index_SMA200"]):
            return "Unavailable — insufficient index history"
        if row["Index_Close"] > row["Index_SMA200"] and row["Index_SMA50"] > row["Index_SMA200"]:
            return "Bull Trend"
        if row["Index_Close"] < row["Index_SMA200"] and row["Index_SMA50"] < row["Index_SMA200"]:
            return "Bear Trend"
        return "Neutral"

    idx["Market_Regime"] = idx.apply(_classify, axis=1)
    return idx


index_features_df = None
if 'index_raw_df' in dir() and index_raw_df is not None:
    try:
        _index_clean = validate_and_clean_ohlc(index_raw_df, min_rows=MIN_ROWS_PRIMARY)
        index_features_df = calculate_index_features(_index_clean)
        print(f"Benchmark index features computed: {len(index_features_df)} rows. "
              f"Latest regime: {index_features_df['Market_Regime'].iloc[-1]}")
    except Exception as e:
        logger.warning(f"Could not build benchmark index features: {type(e).__name__}: {e}")
        index_features_df = None
else:
    print("No benchmark index data available — Market Regime / Relative Strength will be NaN this run.")

In [16]:
# ============================================================
# SECTION 8e — RELATIVE STRENGTH vs. benchmark index
# ============================================================
def calculate_relative_strength(df: pd.DataFrame, index_feat: pd.DataFrame | None) -> pd.DataFrame:
    """
    RS_ND = Stock_Return_ND - Index_Return_ND. Diagnostic/ranking only; NaN (with an explicit
    reason) when either the stock or the index lacks enough history, or when the index wasn't
    available this run — never silently zero-filled.
    """
    rs = pd.DataFrame(index=df.index)
    close = df["Close"]
    rs["Stock_Return_20D"] = (close / close.shift(20) - 1) * 100.0
    rs["Stock_Return_60D"] = (close / close.shift(60) - 1) * 100.0

    if index_feat is None:
        rs["Index_Return_20D"] = np.nan
        rs["Index_Return_60D"] = np.nan
        rs["RS_20D"] = np.nan
        rs["RS_60D"] = np.nan
        rs["Market_Regime"] = "Unavailable — no benchmark index data this run"
        return rs

    aligned = index_feat.reindex(df.index)  # align by trading date; NaN where index has no bar
    rs["Index_Return_20D"] = aligned["Index_Return_20D"]
    rs["Index_Return_60D"] = aligned["Index_Return_60D"]
    rs["RS_20D"] = rs["Stock_Return_20D"] - rs["Index_Return_20D"]
    rs["RS_60D"] = rs["Stock_Return_60D"] - rs["Index_Return_60D"]
    rs["Market_Regime"] = aligned["Market_Regime"]
    return rs

In [17]:
# ============================================================
# SECTION 8f — BREAKOUT STATE MACHINE + EXTENSION CLASSIFICATION
# ============================================================
def calculate_breakout_state(df: pd.DataFrame, bb: pd.DataFrame) -> pd.DataFrame:
    """
    Breakout_Type (expanded from v1's binary Fresh/Continuation):
        Fresh Breakout               : prev Close <= prev Upper_BB AND curr Close > curr Upper_BB
        Continuation                 : prev Close >  prev Upper_BB AND curr Close > curr Upper_BB
        Failed Breakout               : prev Close >  prev Upper_BB AND curr Close <= curr Upper_BB
        No Breakout                  : anything else (never was / isn't above the band)
    Days_Above_Upper_BB: consecutive-day counter of Close > Upper_BB ending at the current row
    (0 when not currently above).

    Extension classification (magnitude only — NOT a claim that any bucket is "better"; see the
    Ranking section for why overshoot-closer-to-4% is deliberately NOT rewarded):
        Controlled : 0% <= overshoot < 1%
        Strong     : 1% <= overshoot < 2%
        Extended   : 2% <= overshoot <= 4%
        Excessive  : overshoot > 4%  (fails the mandatory rule, but still classified for diagnostics)
        N/A        : Close is not above the Upper Band at all
    """
    bo = pd.DataFrame(index=df.index)
    close = df["Close"]
    upper = bb["BB_Upper"]

    is_above = close > upper
    prev_close = close.shift(1)
    prev_upper = upper.shift(1)
    was_above = prev_close > prev_upper

    conditions = [
        (~was_above) & is_above,
        was_above & is_above,
        was_above & (~is_above),
    ]
    choices = ["Fresh Breakout", "Continuation", "Failed Breakout"]
    bo["Breakout_Type"] = np.select(conditions, choices, default="No Breakout")

    # Consecutive days above the upper band, ending at each row.
    streak_id = (~is_above).cumsum()
    bo["Days_Above_Upper_BB"] = is_above.groupby(streak_id).cumsum().where(is_above, 0).astype(int)

    overshoot_pct = (close - upper) / upper * 100.0
    ext_conditions = [
        ~is_above,
        overshoot_pct < 1.0,
        overshoot_pct < 2.0,
        overshoot_pct <= 4.0,
    ]
    ext_choices = ["N/A (Not Above Band)", "Controlled", "Strong", "Extended"]
    bo["Extension_Class"] = np.select(ext_conditions, ext_choices, default="Excessive")

    return bo

## Master feature assembly

Combines every engine above into one per-stock feature table. The three **mandatory** fields
(`BB_Overshoot_Pct`, `HA_Body_Pct`, `HA_Bullish`) are computed exactly as in v1 — nothing about
their arithmetic changes here.

In [18]:
# ============================================================
# SECTION 8g — MASTER FEATURE ASSEMBLY (one function per stock, ties every engine together)
# ============================================================
def calculate_features(df: pd.DataFrame, index_feat: pd.DataFrame | None = None) -> pd.DataFrame:
    bb = calculate_bollinger_bands(df)
    ha = calculate_heikin_ashi(df)
    atr14 = calculate_atr(df, period=14)
    cq = calculate_candle_quality(df)
    vl = calculate_volume_liquidity(df)
    ts = calculate_trend_structure(df)
    rs = calculate_relative_strength(df, index_feat)

    full = df.join([bb, ha, cq, vl, ts, rs])
    full["ATR14"] = atr14
    full["ATR_Pct"] = full["ATR14"] / full["Close"] * 100.0

    # --- MANDATORY fields (unchanged arithmetic from v1) ---
    full["BB_Overshoot_Pct"] = (full["Close"] - full["BB_Upper"]) / full["BB_Upper"] * 100.0
    full["HA_Body_Pct"] = (full["HA_Close"] - full["HA_Open"]) / full["HA_Open"] * 100.0
    full["HA_Bullish"] = full["HA_Close"] > full["HA_Open"]

    # --- ATR-normalized overshoot (diagnostic; Part 9) ---
    full["BB_Overshoot_ATR"] = np.where(
        full["ATR14"] > 0, (full["Close"] - full["BB_Upper"]) / full["ATR14"], np.nan
    )

    # --- Breakout state machine + extension classification (needs bb, computed after merge) ---
    bo = calculate_breakout_state(df, bb)
    full = full.join(bo)

    return full

## Signal engine

The three mandatory conditions are combined exactly as in v1. Every optional filter listed in the
spec is available, independently switchable, and layered strictly **after** the mandatory check —
enabling one never changes what the mandatory check itself evaluates to.

In [19]:
# ============================================================
# SECTION 9 — SIGNAL EVALUATION (mandatory rule, unchanged, + independently-switchable optional filters)
# ============================================================
def evaluate_signal(last_row: pd.Series, prev_row: pd.Series | None):
    bb_breakout = bool(last_row["Close"] > last_row["BB_Upper"])
    bb_size_ok = bool(0 < last_row["BB_Overshoot_Pct"] <= MAX_BB_OVERSHOOT_PCT)
    ha_strength_ok = bool(last_row["HA_Body_Pct"] >= MIN_HA_BODY_PCT)
    mandatory_pass = bb_breakout and bb_size_ok and ha_strength_ok

    optional_checks = {}
    optional_pass = True

    def _apply(flag_name, enabled, ok):
        nonlocal optional_pass
        if enabled:
            optional_checks[flag_name] = bool(ok)
            optional_pass = optional_pass and bool(ok)

    _apply("trend_sma50", USE_TREND_FILTER,
            pd.notna(last_row["SMA50"]) and last_row["Close"] > last_row["SMA50"])
    _apply("long_trend_sma200", USE_LONG_TREND_FILTER,
            pd.notna(last_row["SMA200"]) and last_row["Close"] > last_row["SMA200"])
    _apply("volume_confirmation", USE_VOLUME_FILTER,
            pd.notna(last_row["Volume_Ratio_20"]) and last_row["Volume_Ratio_20"] > 1.0)

    bandwidth_ok = False
    if prev_row is not None and pd.notna(last_row["BB_Width_Pct"]) and pd.notna(prev_row["BB_Width_Pct"]):
        bandwidth_ok = last_row["BB_Width_Pct"] > prev_row["BB_Width_Pct"]
    _apply("bandwidth_expansion", USE_BANDWIDTH_FILTER, bandwidth_ok)

    _apply("atr_normalized_overshoot", USE_ATR_FILTER,
            pd.notna(last_row["BB_Overshoot_ATR"]) and last_row["BB_Overshoot_ATR"] <= ATR_OVERSHOOT_MAX)
    _apply("candle_quality_clv", USE_CANDLE_QUALITY_FILTER,
            pd.notna(last_row["Close_Location_Value"]) and last_row["Close_Location_Value"] >= CANDLE_QUALITY_MIN_CLV)
    _apply("relative_strength", USE_RELATIVE_STRENGTH_FILTER,
            pd.notna(last_row["RS_20D"]) and last_row["RS_20D"] > 0)
    _apply("market_regime_bull", USE_MARKET_REGIME_FILTER,
            last_row.get("Market_Regime") == "Bull Trend")
    _apply("fresh_breakout_only", USE_FRESH_BREAKOUT_ONLY,
            last_row["Breakout_Type"] == "Fresh Breakout")

    final_signal = mandatory_pass and optional_pass

    return {
        "BB_Condition": "PASS" if bb_breakout else "FAIL",
        "BB_Size_Condition": "PASS" if bb_size_ok else "FAIL",
        "HA_Condition": "PASS" if ha_strength_ok else "FAIL",
        "Mandatory_Pass": mandatory_pass,
        "Optional_Checks": optional_checks,
        "Optional_Pass": optional_pass,
        "Signal": final_signal,
    }

In [20]:
# ============================================================
# SECTION 10 — SANITY / VALIDATION CHECKS — fail loudly on logic bugs, never silently corrupt output
# ============================================================
def sanity_check_feature_row(row: pd.Series):
    if pd.notna(row["BB_Upper"]) and pd.notna(row["BB_Middle"]) and pd.notna(row["BB_Lower"]):
        assert row["BB_Upper"] >= row["BB_Middle"] >= row["BB_Lower"], "BB ordering violated"
    assert row["HA_High"] >= row["HA_Open"] - 1e-9, "HA_High < HA_Open"
    assert row["HA_High"] >= row["HA_Close"] - 1e-9, "HA_High < HA_Close"
    assert row["HA_Low"] <= row["HA_Open"] + 1e-9, "HA_Low > HA_Open"
    assert row["HA_Low"] <= row["HA_Close"] + 1e-9, "HA_Low > HA_Close"


def sanity_check_pass_row(row: pd.Series):
    assert row["Close"] > row["BB_Upper"], "PASS row fails Close > BB_Upper"
    assert 0 < row["BB_Overshoot_Pct"] <= MAX_BB_OVERSHOOT_PCT, "PASS row fails overshoot bound"
    assert row["HA_Body_Pct"] >= MIN_HA_BODY_PCT, "PASS row fails HA body threshold"
    mandatory_fields = ["Close", "BB_Upper", "BB_Overshoot_Pct", "HA_Open", "HA_Close", "HA_Body_Pct"]
    assert row[mandatory_fields].notna().all(), "PASS row has NaN in a mandatory signal field"

## Per-stock pipeline & universe loop

One bad ticker never halts the run. Every failure is logged with `ticker`, `stage`,
`exception_type`, `exception_message`, and a timestamp. If `RUN_RESEARCH_MODE=True`, the full
per-stock feature history (not just the last row) is retained in memory for the event study below.

In [21]:
# ============================================================
# SECTION 11 — scan_stock(): full pipeline for a single ticker
# ============================================================
stock_feature_frames = {}   # nse_symbol -> full feature DataFrame, populated only if RUN_RESEARCH_MODE


def scan_stock(nse_symbol: str, yf_symbol: str, company_name: str, sector, mapping_method,
               raw_df, index_feat, now_ist: datetime):
    result = {
        "Stock": nse_symbol,
        "YFinance_Symbol": yf_symbol,
        "Mapping_Method": mapping_method,
        "Mapping_Status": "ERROR",
        "Company_Name": company_name,
        "Sector": sector,
        "status": "ERROR",
        "stage": None,
        "timestamp": now_ist.strftime("%Y-%m-%d %H:%M:%S"),
        "rows_downloaded": 0 if raw_df is None else len(raw_df),
        "last_data_date": None,
        "candle_complete": None,
        "was_trimmed": None,
        "data_stale": None,
        "stale_age_days": None,
        "error_message": None,
    }

    try:
        result["stage"] = "download"
        if raw_df is None:
            raise ValueError("no data downloaded for this symbol")

        result["stage"] = "validation"
        clean_df = validate_and_clean_ohlc(raw_df)

        clean_df, was_trimmed = trim_incomplete_candle(clean_df, now_ist)
        if len(clean_df) < MIN_ROWS_PRIMARY:
            raise ValueError(f"insufficient history after trimming incomplete candle: {len(clean_df)} rows")

        is_stale, stale_age_days = assess_staleness(clean_df, now_ist)

        result["stage"] = "indicators"
        features = calculate_features(clean_df, index_feat)
        last_row = features.iloc[-1]
        prev_row = features.iloc[-2] if len(features) >= 2 else None

        result["stage"] = "indicators"  # (sanity checks are part of the indicator-verification stage)
        sanity_check_feature_row(last_row)

        result["stage"] = "signal"
        signal = evaluate_signal(last_row, prev_row)

        if signal["Signal"]:
            sanity_check_pass_row(last_row)

        if RUN_RESEARCH_MODE:
            stock_feature_frames[nse_symbol] = features

        result.update({
            "status": "OK",
            "Mapping_Status": "OK",
            "stage": "complete",
            "rows_downloaded": len(raw_df),
            "rows_valid": len(clean_df),
            "last_data_date": clean_df.index[-1].strftime("%Y-%m-%d"),
            "candle_complete": True,
            "was_trimmed": was_trimmed,
            "data_stale": is_stale,
            "stale_age_days": stale_age_days,
            "Signal_Date": clean_df.index[-1].strftime("%Y-%m-%d"),
            "CMP": float(last_row["Close"]),
            "BB_Middle": _f(last_row["BB_Middle"]),
            "BB_Upper": _f(last_row["BB_Upper"]),
            "BB_Lower": _f(last_row["BB_Lower"]),
            "BB_Width_Pct": _f(last_row["BB_Width_Pct"]),
            "BB_PctB": _f(last_row["BB_PctB"]),
            "BB_Overshoot_Pct": _f(last_row["BB_Overshoot_Pct"]),
            "HA_Open": float(last_row["HA_Open"]),
            "HA_Close": float(last_row["HA_Close"]),
            "HA_Body_Pct": _f(last_row["HA_Body_Pct"]),
            "HA_Upper_Wick_Pct": _f(last_row["HA_Upper_Wick_Pct"]),
            "HA_Lower_Wick_Pct": _f(last_row["HA_Lower_Wick_Pct"]),
            "HA_Bullish": bool(last_row["HA_Bullish"]),
            "ATR14": _f(last_row["ATR14"]),
            "ATR_Pct": _f(last_row["ATR_Pct"]),
            "BB_Overshoot_ATR": _f(last_row["BB_Overshoot_ATR"]),
            "Candle_Body_Pct": _f(last_row["Candle_Body_Pct"]),
            "Close_Location_Value": _f(last_row["Close_Location_Value"]),
            "Volume": float(last_row["Volume"]),
            "Avg_Volume_20": _f(last_row["Avg_Volume_20"]),
            "Volume_Ratio_20": _f(last_row["Volume_Ratio_20"]),
            "Dollar_Volume": _f(last_row["Dollar_Volume"]),
            "SMA20": _f(last_row["SMA20"]),
            "SMA50": _f(last_row["SMA50"]),
            "SMA200": _f(last_row["SMA200"]),
            "SMA50_Status": last_row["SMA50_Status"],
            "SMA200_Status": last_row["SMA200_Status"],
            "SMA20_Slope_10D": _f(last_row["SMA20_Slope_10D"]),
            "SMA50_Slope_20D": _f(last_row["SMA50_Slope_20D"]),
            "SMA200_Slope_20D": _f(last_row["SMA200_Slope_20D"]),
            "Dist_SMA50_Pct": _f(last_row["Dist_SMA50_Pct"]),
            "Dist_SMA200_Pct": _f(last_row["Dist_SMA200_Pct"]),
            "RS_20D": _f(last_row["RS_20D"]),
            "RS_60D": _f(last_row["RS_60D"]),
            "Market_Regime": last_row.get("Market_Regime"),
            "Breakout_Type": str(last_row["Breakout_Type"]),
            "Days_Above_Upper_BB": int(last_row["Days_Above_Upper_BB"]),
            "Extension_Class": str(last_row["Extension_Class"]),
            "BB_Condition": signal["BB_Condition"],
            "BB_Size_Condition": signal["BB_Size_Condition"],
            "HA_Condition": signal["HA_Condition"],
            "Mandatory_Pass": signal["Mandatory_Pass"],
            "Optional_Pass": signal["Optional_Pass"],
            "Signal": signal["Signal"],
        })
        return result

    except Exception as e:
        result["error_message"] = f"{type(e).__name__}: {e}"
        result["Mapping_Status"] = "OK" if result["stage"] not in ("download",) else "NO_DATA"
        _stage, _err = result["stage"], result["error_message"]
        logger.warning(f"{nse_symbol}: FAILED at stage='{_stage}' -> {_err}")
        return result


def _f(x):
    return float(x) if pd.notna(x) else np.nan

In [22]:
# ============================================================
# SECTION 12 — scan_universe(): loop over every constituent with progress logging
# ============================================================
def scan_universe(constituents_df: pd.DataFrame, raw_data: dict, index_feat, now_ist: datetime):
    results = []
    total = len(constituents_df)

    print("NIFTY 200 scanner started")
    print(f"Universe size: {total}\n")

    for i, row in enumerate(constituents_df.itertuples(index=False), start=1):
        print(f"Processing {i}/{total}: {row.Symbol}")
        raw_df = raw_data.get(row.YFinance_Symbol)
        res = scan_stock(row.Symbol, row.YFinance_Symbol, row.Company_Name, row.Sector,
                          row.Mapping_Method, raw_df, index_feat, now_ist)
        results.append(res)

    print("\nScan complete")
    return pd.DataFrame(results)


scan_results_df = scan_universe(constituents_df, raw_ohlc_data, index_features_df, RUN_TIMESTAMP_IST)

# Feed Mapping_Status back into the universe table for the "Universe" audit sheet.
_status_map = scan_results_df.set_index("Stock")["Mapping_Status"].to_dict()
constituents_df["Mapping_Status"] = constituents_df["Symbol"].map(_status_map).fillna("NOT_SCANNED")

n_total_constituents = len(constituents_df)
n_scanned_ok = int((scan_results_df["status"] == "OK").sum())
n_failed = int((scan_results_df["status"] == "ERROR").sum())
n_passing = int((scan_results_df["Signal"] == True).sum()) if "Signal" in scan_results_df.columns else 0

print(f"\nTotal constituents        : {n_total_constituents}")
print(f"Successfully scanned      : {n_scanned_ok}")
print(f"Failed                    : {n_failed}")
print(f"Primary-condition matches : {n_passing}")

if n_failed:
    print("\nFailure stage breakdown:")
    print(scan_results_df[scan_results_df["status"] == "ERROR"]["stage"].value_counts())

NIFTY 200 scanner started
Universe size: 200

Processing 1/200: 360ONE
Processing 2/200: ABB
Processing 3/200: APLAPOLLO
Processing 4/200: AUBANK
Processing 5/200: ADANIENSOL
Processing 6/200: ADANIENT
Processing 7/200: ADANIGREEN
Processing 8/200: ADANIPORTS
Processing 9/200: ADANIPOWER
Processing 10/200: ATGL
Processing 11/200: ABCAPITAL
Processing 12/200: ALKEM
Processing 13/200: AMBUJACEM
Processing 14/200: APOLLOHOSP
Processing 15/200: ASHOKLEY
Processing 16/200: ASIANPAINT
Processing 17/200: ASTRAL
Processing 18/200: AUROPHARMA
Processing 19/200: DMART
Processing 20/200: AXISBANK
Processing 21/200: BSE
Processing 22/200: BAJAJ-AUTO
Processing 23/200: BAJFINANCE
Processing 24/200: BAJAJFINSV
Processing 25/200: BAJAJHLDNG
Processing 26/200: BANKBARODA
Processing 27/200: BANKINDIA
Processing 28/200: BDL
Processing 29/200: BEL
Processing 30/200: BHARATFORG
Processing 31/200: BHEL
Processing 32/200: BPCL
Processing 33/200: BHARTIARTL
Processing 34/200: GROWW
Processing 35/200: BIOCON


## Ranking — Research Heuristic Score (audit fix: no more "closer to 4% = better")

v1's score gave more points as overshoot approached 4%, which the audit correctly called out as an
unjustified assumption (a 3.9% overshoot is not automatically a better setup than a 0.8% one — it
can just as easily represent exhaustion). This version removes that bias entirely: overshoot
magnitude is *classified* (`Extension_Class`) but does **not** feed the score in either direction.
The score is explicitly labeled a **Research Heuristic Score** — not a probability, not an expected
return, not a validated edge.

In [23]:
# ============================================================
# SECTION 13 — SIGNAL QUALITY SCORE ("Research Heuristic Score" — diagnostic/ranking ONLY)
# ============================================================
def _clip01(x, lo, hi):
    if pd.isna(x):
        return 0.5   # neutral midpoint when unknown, rather than silently punishing missing data
    return float(np.clip((x - lo) / (hi - lo), 0.0, 1.0))


def _trend_score(row) -> float:
    """0-100. Rewards price above SMA50/SMA200 with SMA50>SMA200 and positive slopes. Purely
    structural — does not use overshoot magnitude at all."""
    pts, n = 0.0, 0
    cmp_ = row.get("CMP", row.get("Close"))
    if pd.notna(row.get("SMA50")):
        pts += 100.0 if cmp_ > row["SMA50"] else 0.0; n += 1
    if pd.notna(row.get("SMA200")):
        pts += 100.0 if cmp_ > row["SMA200"] else 0.0; n += 1
    if pd.notna(row.get("SMA50")) and pd.notna(row.get("SMA200")):
        pts += 100.0 if row["SMA50"] > row["SMA200"] else 0.0; n += 1
    if pd.notna(row.get("SMA50_Slope_20D")):
        pts += 100.0 if row["SMA50_Slope_20D"] > 0 else 0.0; n += 1
    return pts / n if n else 50.0


def _momentum_score(row) -> float:
    """0-100 from RS_20D/RS_60D (relative strength vs. benchmark), NOT from BB overshoot size."""
    vals = []
    if pd.notna(row.get("RS_20D")):
        vals.append(_clip01(row["RS_20D"], -10, 10) * 100)
    if pd.notna(row.get("RS_60D")):
        vals.append(_clip01(row["RS_60D"], -15, 15) * 100)
    return float(np.mean(vals)) if vals else 50.0


def _volatility_score(row) -> float:
    """0-100, peaks for a MODERATE ATR-normalized overshoot (not "bigger is better", not
    "smaller is better") — a trapezoid centered on a defensible mid-range, explicitly arbitrary
    and documented as such rather than silently baked in."""
    x = row.get("BB_Overshoot_ATR")
    if pd.isna(x):
        return 50.0
    # peak plateau between 0.2x and 1.0x ATR; decays toward 0 by 0.0x and by 2.5x
    if x <= 0:
        return 0.0
    if x < 0.2:
        return float(np.clip(x / 0.2, 0, 1) * 100)
    if x <= 1.0:
        return 100.0
    if x <= 2.5:
        return float(np.clip(1 - (x - 1.0) / 1.5, 0, 1) * 100)
    return 0.0


def _candle_quality_score(row) -> float:
    """0-100 from Close_Location_Value (close near the high = higher score)."""
    clv = row.get("Close_Location_Value")
    if pd.isna(clv):
        return 50.0
    return float(np.clip((clv + 1) / 2, 0, 1) * 100)   # CLV in [-1,1] -> [0,100]


def _liquidity_score(row, dollar_volume_percentile: float | None) -> float:
    """0-100 from cross-sectional Dollar_Volume percentile within today's scanned universe (filled
    in as a post-processing step — see below), NOT from a single fixed volume threshold."""
    if dollar_volume_percentile is None or pd.isna(dollar_volume_percentile):
        return 50.0
    return float(np.clip(dollar_volume_percentile, 0, 1) * 100)


def _market_regime_score(row) -> float:
    """0-100, simplistic explicit mapping (documented, not tuned): Bull=100, Neutral=50, Bear=0."""
    regime = row.get("Market_Regime")
    return {"Bull Trend": 100.0, "Neutral": 50.0, "Bear Trend": 0.0}.get(regime, 50.0)


def compute_research_heuristic_score(row, dollar_volume_percentile=None) -> dict:
    components = {
        "Trend_Score": round(_trend_score(row), 1),
        "Momentum_Score": round(_momentum_score(row), 1),
        "Volatility_Score": round(_volatility_score(row), 1),
        "Candle_Quality_Score": round(_candle_quality_score(row), 1),
        "Liquidity_Score": round(_liquidity_score(row, dollar_volume_percentile), 1),
        "Market_Regime_Score": round(_market_regime_score(row), 1),
    }
    overall = sum(components[k] * SCORE_WEIGHTS[k] for k in SCORE_WEIGHTS)
    components["Research_Heuristic_Score"] = round(overall, 2)
    return components

In [24]:
# ============================================================
# SECTION 14 — SIGNAL_REASON (auditable, plain-language, no unsupported probability claims)
# ============================================================
def build_signal_reason(row: pd.Series) -> str:
    parts = [
        f"Close Rs {row['CMP']:,.2f} is {row['BB_Overshoot_Pct']:.2f}% above the Upper Bollinger Band "
        f"({row['Extension_Class']}).",
        f"HA body is {row['HA_Body_Pct']:.2f}% (HA Close above HA Open).",
        f"Breakout type: {row['Breakout_Type']}, {row['Days_Above_Upper_BB']} day(s) above the upper band.",
    ]
    if pd.notna(row.get("Market_Regime")) and row.get("Market_Regime") not in (None, "nan"):
        parts.append(f"Benchmark ({INDEX_NAME}) regime: {row['Market_Regime']}.")
    if pd.notna(row.get("Volume_Ratio_20")):
        parts.append(f"Volume is {row['Volume_Ratio_20']:.2f}x its 20-day average.")
    if pd.notna(row.get("Dist_SMA50_Pct")) and pd.notna(row.get("Dist_SMA200_Pct")):
        parts.append(f"Close is {row['Dist_SMA50_Pct']:.1f}% vs SMA50 and {row['Dist_SMA200_Pct']:.1f}% vs SMA200.")
    parts.append("All mandatory conditions passed. This is a technical screening result, not a probability estimate.")
    return " ".join(parts)

## Results assembly

Percentages are still computed internally as "percentage points" (e.g. `2.31` meaning 2.31%) —
that convention doesn't change. The Excel-format bug (Part 29) is fixed at the **export** step
below, where these columns are converted to fractions specifically for the cells Excel renders
with a `%` number format. Working in percentage points internally keeps every threshold comparison
(`MAX_BB_OVERSHOOT_PCT = 4.0`, etc.) exactly as originally specified.

In [25]:
# ============================================================
# SECTION 15 — RESULTS ASSEMBLY: cross-sectional percentile, score, Signals/Diagnostics/Universe
# ============================================================
ok_df = scan_results_df[scan_results_df["status"] == "OK"].copy()

# Cross-sectional Dollar_Volume percentile within TODAY'S successfully-scanned universe (used only
# by the Liquidity_Score component — never to reject a stock).
if len(ok_df) > 0:
    ok_df["Dollar_Volume_Percentile"] = ok_df["Dollar_Volume"].rank(pct=True)
else:
    ok_df["Dollar_Volume_Percentile"] = np.nan

score_rows = ok_df.apply(
    lambda r: compute_research_heuristic_score(r, r["Dollar_Volume_Percentile"]), axis=1, result_type="expand"
)
ok_df = pd.concat([ok_df, score_rows], axis=1)

passing_df = ok_df[ok_df["Signal"] == True].copy()

if not passing_df.empty:
    passing_df["Signal_Reason"] = passing_df.apply(build_signal_reason, axis=1)
    passing_df = passing_df.sort_values("Research_Heuristic_Score", ascending=False).reset_index(drop=True)
    passing_df.insert(0, "Rank", range(1, len(passing_df) + 1))

SIGNALS_COLUMNS = [
    "Rank", "Signal_Date", "Stock", "Company_Name", "Sector", "CMP",
    "BB_Middle", "BB_Upper", "BB_Lower", "BB_Width_Pct", "BB_PctB", "BB_Overshoot_Pct",
    "HA_Open", "HA_Close", "HA_Body_Pct", "HA_Upper_Wick_Pct", "HA_Lower_Wick_Pct",
    "ATR14", "ATR_Pct", "BB_Overshoot_ATR",
    "Volume", "Volume_Ratio_20",
    "SMA20", "SMA50", "SMA200", "Dist_SMA50_Pct", "Dist_SMA200_Pct",
    "RS_20D", "RS_60D",
    "Breakout_Type", "Days_Above_Upper_BB", "Market_Regime",
    "Research_Heuristic_Score", "Signal_Reason",
]
signals_df = passing_df[SIGNALS_COLUMNS].copy() if not passing_df.empty else pd.DataFrame(columns=SIGNALS_COLUMNS)

print(f"Stocks passing all mandatory conditions: {len(signals_df)}")
signals_df.head(10)

Stocks passing all mandatory conditions: 3


,Rank,Signal_Date,Stock,Company_Name,Sector,CMP,BB_Middle,BB_Upper,BB_Lower,BB_Width_Pct,BB_PctB,BB_Overshoot_Pct,HA_Open,HA_Close,HA_Body_Pct,HA_Upper_Wick_Pct,HA_Lower_Wick_Pct,ATR14,ATR_Pct,BB_Overshoot_ATR,Volume,Volume_Ratio_20,SMA20,SMA50,SMA200,Dist_SMA50_Pct,Dist_SMA200_Pct,RS_20D,RS_60D,Breakout_Type,Days_Above_Upper_BB,Market_Regime,Research_Heuristic_Score,Signal_Reason
0,1,2026-09-07,SOLARINDS,Solar Industries India Ltd.,Chemicals,21950.00,20195.950000,21769.071132,18622.828868,15.578580,1.057506,0.831128,21079.123857,21762.500000,3.241957,2.075513,0.0,550.642857,2.508623,0.328578,227176.0,1.197564,20195.950000,19146.680469,15685.855293,14.641282,39.934990,NaN,NaN,Fresh Breakout,1,Unavailable — no benchmark index data this run,76.61,"Close Rs 21,950.00 is 0.83% above the Upper Bo..."
1,2,2026-09-08,GVT&D,GE Vernova T&D India Ltd.,Capital Goods,4750.00,4299.738940,4605.837917,3993.639964,14.238026,1.235483,3.129986,4338.009974,4697.275024,8.281794,2.206656,0.0,161.240766,3.394542,0.894080,4274212.0,4.155982,4299.738940,4372.861021,3956.672698,8.624536,20.050365,NaN,NaN,Fresh Breakout,1,Unavailable — no benchmark index data this run,72.72,"Close Rs 4,750.00 is 3.13% above the Upper Bol..."
2,3,2026-09-08,COALINDIA,Coal India Ltd.,Oil Gas & Consumable Fuels,420.25,403.736394,420.067376,387.405411,8.089923,1.005591,0.043475,415.065710,419.500000,1.068334,0.987797,0.0,7.829704,1.863106,0.023324,5273145.0,0.527231,403.736394,410.488101,418.634450,2.378120,0.385909,NaN,NaN,Continuation,4,Unavailable — no benchmark index data this run,47.09,Close Rs 420.25 is 0.04% above the Upper Bolli...


In [26]:
# ============================================================
# SECTION 16 — DIAGNOSTICS DATAFRAME (every successfully scanned stock, PASS or FAIL, auditable)
# ============================================================
diag_source = ok_df.copy()
diag_source["Final_Signal"] = diag_source["Signal"].map({True: "PASS", False: "FAIL"})

DIAGNOSTICS_COLUMNS = [
    "Stock", "last_data_date", "data_stale", "CMP", "BB_Upper", "BB_Overshoot_Pct", "Extension_Class",
    "HA_Open", "HA_Close", "HA_Body_Pct",
    "BB_Condition", "BB_Size_Condition", "HA_Condition", "Mandatory_Pass", "Optional_Pass",
    "Final_Signal", "Breakout_Type", "Days_Above_Upper_BB", "Market_Regime",
    "Research_Heuristic_Score",
]
diagnostics_df = diag_source.rename(columns={"last_data_date": "Last_Data_Date"})[
    [c if c != "last_data_date" else "Last_Data_Date" for c in DIAGNOSTICS_COLUMNS]
].sort_values("BB_Overshoot_Pct", ascending=False)

failed_df = scan_results_df[scan_results_df["status"] == "ERROR"][
    ["Stock", "stage", "error_message"]
].rename(columns={"stage": "Failed_Stage", "error_message": "Error_Status"})

print(f"Diagnostics rows (successfully scanned): {len(diagnostics_df)}")
print(f"Failed rows (excluded, but logged in Scan_Log): {len(failed_df)}")
diagnostics_df.head(10)

Diagnostics rows (successfully scanned): 200
Failed rows (excluded, but logged in Scan_Log): 0


,Stock,Last_Data_Date,data_stale,CMP,BB_Upper,BB_Overshoot_Pct,Extension_Class,HA_Open,HA_Close,HA_Body_Pct,BB_Condition,BB_Size_Condition,HA_Condition,Mandatory_Pass,Optional_Pass,Final_Signal,Breakout_Type,Days_Above_Upper_BB,Market_Regime,Research_Heuristic_Score
61,GVT&D,2026-09-08,False,4750.000000,4605.837917,3.129986,Extended,4338.009974,4697.275024,8.281794,PASS,PASS,PASS,True,True,PASS,Fresh Breakout,1,Unavailable — no benchmark index data this run,72.72
164,SOLARINDS,2026-09-07,False,21950.000000,21769.071132,0.831128,Controlled,21079.123857,21762.500000,3.241957,PASS,PASS,PASS,True,True,PASS,Fresh Breakout,1,Unavailable — no benchmark index data this run,76.61
42,COALINDIA,2026-09-08,False,420.250000,420.067376,0.043475,Controlled,415.065710,419.500000,1.068334,PASS,PASS,PASS,True,True,PASS,Continuation,4,Unavailable — no benchmark index data this run,47.09
109,LICHSGFIN,2026-09-04,False,562.049988,562.727014,-0.120312,N/A (Not Above Band),540.521682,562.362503,4.040693,FAIL,FAIL,PASS,False,True,FAIL,No Breakout,0,Unavailable — no benchmark index data this run,43.41
112,LAURUSLABS,2026-09-08,False,1946.900024,1952.164109,-0.269654,N/A (Not Above Band),1858.938608,1908.100006,2.644595,FAIL,FAIL,PASS,False,True,FAIL,No Breakout,0,Unavailable — no benchmark index data this run,65.39
38,CGPOWER,2026-09-08,False,910.700012,915.024977,-0.472661,N/A (Not Above Band),895.277630,903.150009,0.879323,FAIL,FAIL,FAIL,False,True,FAIL,No Breakout,0,Unavailable — no benchmark index data this run,55.70
156,SBICARD,2026-09-08,False,668.000000,671.281469,-0.488836,N/A (Not Above Band),653.365496,668.837494,2.368046,FAIL,FAIL,PASS,False,True,FAIL,No Breakout,0,Unavailable — no benchmark index data this run,42.70
136,OIL,2026-09-08,False,495.399994,497.872248,-0.496564,N/A (Not Above Band),489.090889,491.474991,0.487456,FAIL,FAIL,FAIL,False,True,FAIL,No Breakout,0,Unavailable — no benchmark index data this run,57.27
17,AUROPHARMA,2026-09-08,False,1692.000000,1702.664163,-0.626322,N/A (Not Above Band),1650.522320,1676.575012,1.578451,FAIL,FAIL,PASS,False,True,FAIL,No Breakout,0,Unavailable — no benchmark index data this run,61.12
51,DIVISLAB,2026-09-08,False,9575.000000,9647.815229,-0.754733,N/A (Not Above Band),9206.796710,9447.875000,2.618482,FAIL,FAIL,PASS,False,True,FAIL,No Breakout,0,Unavailable — no benchmark index data this run,66.52


In [27]:
# ============================================================
# SECTION 17 — UNIVERSE AUDIT SHEET (full mapping trail, not just the passing/failing stocks)
# ============================================================
universe_audit_df = constituents_df[["Symbol", "Company_Name", "Sector", "YFinance_Symbol",
                                      "Mapping_Method", "Mapping_Status"]].rename(
    columns={"Symbol": "NSE_Symbol", "YFinance_Symbol": "YF_Symbol"}
)
print("Mapping status breakdown:")
print(universe_audit_df["Mapping_Status"].value_counts())
universe_audit_df.head(10)

Mapping status breakdown:
Mapping_Status
OK    200
Name: count, dtype: int64


,NSE_Symbol,Company_Name,Sector,YF_Symbol,Mapping_Method,Mapping_Status
0,360ONE,360 ONE WAM Ltd.,Financial Services,360ONE.NS,direct_suffix,OK
1,ABB,ABB India Ltd.,Capital Goods,ABB.NS,direct_suffix,OK
2,APLAPOLLO,APL Apollo Tubes Ltd.,Capital Goods,APLAPOLLO.NS,direct_suffix,OK
3,AUBANK,AU Small Finance Bank Ltd.,Financial Services,AUBANK.NS,direct_suffix,OK
4,ADANIENSOL,Adani Energy Solutions Ltd.,Power,ADANIENSOL.NS,direct_suffix,OK
5,ADANIENT,Adani Enterprises Ltd.,Metals & Mining,ADANIENT.NS,direct_suffix,OK
6,ADANIGREEN,Adani Green Energy Ltd.,Power,ADANIGREEN.NS,direct_suffix,OK
7,ADANIPORTS,Adani Ports and Special Economic Zone Ltd.,Services,ADANIPORTS.NS,direct_suffix,OK
8,ADANIPOWER,Adani Power Ltd.,Power,ADANIPOWER.NS,direct_suffix,OK
9,ATGL,Adani Total Gas Ltd.,Oil Gas & Consumable Fuels,ATGL.NS,direct_suffix,OK


## Scan log & market regime summary

In [28]:
# ============================================================
# SECTION 18 — SCAN LOG (every ticker, every stage, every failure — auditable)
# ============================================================
scan_log_df = scan_results_df[[
    "Stock", "YFinance_Symbol", "timestamp", "status", "stage", "rows_downloaded",
    "last_data_date", "data_stale", "stale_age_days", "was_trimmed", "error_message",
]].rename(columns={"YFinance_Symbol": "yfinance_symbol", "stage": "last_stage_reached"})
scan_log_df.head(10)

,Stock,yfinance_symbol,timestamp,status,last_stage_reached,rows_downloaded,last_data_date,data_stale,stale_age_days,was_trimmed,error_message
0,360ONE,360ONE.NS,2026-09-09 08:55:29,OK,complete,500,2026-09-08,False,1,False,None
1,ABB,ABB.NS,2026-09-09 08:55:29,OK,complete,500,2026-09-08,False,1,False,None
2,APLAPOLLO,APLAPOLLO.NS,2026-09-09 08:55:29,OK,complete,500,2026-09-07,False,2,False,None
3,AUBANK,AUBANK.NS,2026-09-09 08:55:29,OK,complete,500,2026-09-04,False,5,False,None
4,ADANIENSOL,ADANIENSOL.NS,2026-09-09 08:55:29,OK,complete,500,2026-09-08,False,1,False,None
5,ADANIENT,ADANIENT.NS,2026-09-09 08:55:29,OK,complete,500,2026-09-07,False,2,False,None
6,ADANIGREEN,ADANIGREEN.NS,2026-09-09 08:55:29,OK,complete,500,2026-09-07,False,2,False,None
7,ADANIPORTS,ADANIPORTS.NS,2026-09-09 08:55:29,OK,complete,500,2026-09-08,False,1,False,None
8,ADANIPOWER,ADANIPOWER.NS,2026-09-09 08:55:29,OK,complete,500,2026-09-08,False,1,False,None
9,ATGL,ATGL.NS,2026-09-09 08:55:29,OK,complete,500,2026-09-08,False,1,False,None


In [29]:
# ============================================================
# SECTION 19 — MARKET REGIME SUMMARY SHEET
# ============================================================
if index_features_df is not None and len(index_features_df) > 0:
    _latest_idx = index_features_df.iloc[-1]
    market_regime_summary_df = pd.DataFrame([{
        "Benchmark_Index": f"{INDEX_TICKER} ({INDEX_NAME})",
        "Latest_Date": index_features_df.index[-1].strftime("%Y-%m-%d"),
        "Index_Close": round(float(_latest_idx["Index_Close"]), 2),
        "Index_SMA20": round(float(_latest_idx["Index_SMA20"]), 2) if pd.notna(_latest_idx["Index_SMA20"]) else np.nan,
        "Index_SMA50": round(float(_latest_idx["Index_SMA50"]), 2) if pd.notna(_latest_idx["Index_SMA50"]) else np.nan,
        "Index_SMA200": round(float(_latest_idx["Index_SMA200"]), 2) if pd.notna(_latest_idx["Index_SMA200"]) else np.nan,
        "Index_Return_20D_Pct": round(float(_latest_idx["Index_Return_20D"]), 2) if pd.notna(_latest_idx["Index_Return_20D"]) else np.nan,
        "Index_Return_60D_Pct": round(float(_latest_idx["Index_Return_60D"]), 2) if pd.notna(_latest_idx["Index_Return_60D"]) else np.nan,
        "Market_Regime": _latest_idx["Market_Regime"],
        "Regime_Rule": "Bull: Close>SMA200 & SMA50>SMA200 | Bear: Close<SMA200 & SMA50<SMA200 | else Neutral",
    }])
    print(f"Current market regime ({INDEX_NAME}): {_latest_idx['Market_Regime']}")
else:
    market_regime_summary_df = pd.DataFrame([{
        "Benchmark_Index": f"{INDEX_TICKER} ({INDEX_NAME})",
        "Latest_Date": None, "Index_Close": np.nan, "Index_SMA20": np.nan, "Index_SMA50": np.nan,
        "Index_SMA200": np.nan, "Index_Return_20D_Pct": np.nan, "Index_Return_60D_Pct": np.nan,
        "Market_Regime": "Unavailable — benchmark index data not retrieved this run",
        "Regime_Rule": "Bull: Close>SMA200 & SMA50>SMA200 | Bear: Close<SMA200 & SMA50<SMA200 | else Neutral",
    }])
    print("Market regime unavailable this run (no benchmark index data).")

market_regime_summary_df

Market regime unavailable this run (no benchmark index data).


,Benchmark_Index,Latest_Date,Index_Close,Index_SMA20,Index_SMA50,Index_SMA200,Index_Return_20D_Pct,Index_Return_60D_Pct,Market_Regime,Regime_Rule
0,^NSEI (NIFTY 50),None,NaN,NaN,NaN,NaN,NaN,NaN,Unavailable — benchmark index data not retriev...,Bull: Close>SMA200 & SMA50>SMA200 | Bear: Clos...


## Historical event study (optional — `RUN_RESEARCH_MODE`)

**Answers "what happened historically after this exact setup?" instead of only "which stocks
qualify today?"** Uses the full per-stock history already downloaded (no extra network calls).

**Survivorship bias disclosure (Part 20 / audit point 19):** this notebook only has *today's*
NIFTY 200 membership, not historical point-in-time membership. Running the mandatory conditions
back across today's constituents' own history is therefore explicitly labeled a
**"Current-Universe Historical Simulation"** — stocks that were removed from the index (often after
underperforming) are absent from this history by construction, which tends to bias results
optimistic. This is disclosed here and repeated in the Research_Summary Excel sheet; it is not an
unbiased point-in-time NIFTY 200 backtest.

**Entry timing:** a signal detected at the completed close of day T is entered at day T+1's
`ENTRY_PRICE_METHOD` (default: next day's **Open**) — never at day T's own close, which would be
look-ahead (the close of T is exactly the bar that produced the signal).

In [30]:
# ============================================================
# SECTION 20 — HISTORICAL EVENT EXTRACTION (vectorized signal scan across full history)
# ============================================================
MAX_HORIZON = max(FORWARD_RETURN_HORIZONS)


def extract_historical_events(feature_frames: dict) -> pd.DataFrame:
    """
    For every stock's full feature history, finds every historical date where the REGULAR close
    was above the Upper Bollinger Band (the broadest possible candidate set — any overshoot
    magnitude, any HA body), then computes forward returns / MFE / MAE from a clearly-defined
    entry point. No look-ahead: entry is always at T+1, using only information available at/after
    T+1.

    Returning the BROAD candidate set (not yet filtered to MAX_BB_OVERSHOOT_PCT / MIN_HA_BODY_PCT)
    lets the sensitivity analysis later test other threshold combinations by filtering this same
    precomputed set, instead of recomputing forward returns for every parameter combination.
    The official mandatory-condition event set is derived by filtering this candidate set — see
    the cell below.
    """
    events = []

    for stock, feat in feature_frames.items():
        n = len(feat)
        close = feat["Close"].values
        open_ = feat["Open"].values
        high = feat["High"].values
        low = feat["Low"].values

        bb_breakout = (feat["Close"] > feat["BB_Upper"]).values
        signal_positions = np.where(bb_breakout)[0]

        for pos in signal_positions:
            entry_pos = pos + 1
            if entry_pos >= n:
                continue   # signal is on the latest available bar — no forward data exists yet

            entry_price = open_[entry_pos] if ENTRY_PRICE_METHOD == "next_open" else close[entry_pos]
            if not np.isfinite(entry_price) or entry_price <= 0:
                continue

            row = feat.iloc[pos]
            event = {
                "Stock": stock,
                "Signal_Date": feat.index[pos].strftime("%Y-%m-%d"),
                "Entry_Date": feat.index[entry_pos].strftime("%Y-%m-%d"),
                "Entry_Price": float(entry_price),
                "Entry_Method": ENTRY_PRICE_METHOD,
                "BB_Overshoot_Pct": float(row["BB_Overshoot_Pct"]),
                "HA_Body_Pct": float(row["HA_Body_Pct"]),
                "Extension_Class": row["Extension_Class"],
                "Breakout_Type": row["Breakout_Type"],
                "Market_Regime": row.get("Market_Regime"),
                "ATR_Pct": float(row["ATR_Pct"]) if pd.notna(row["ATR_Pct"]) else np.nan,
                "BB_Overshoot_ATR": float(row["BB_Overshoot_ATR"]) if pd.notna(row["BB_Overshoot_ATR"]) else np.nan,
                "Volume_Ratio_20": float(row["Volume_Ratio_20"]) if pd.notna(row["Volume_Ratio_20"]) else np.nan,
                # Flags used by the A/B research table — each is exactly the corresponding
                # OPTIONAL filter's condition, evaluated AT SIGNAL TIME (no look-ahead):
                "Trend_Flag": bool(pd.notna(row["SMA50"]) and row["Close"] > row["SMA50"]),
                "Volume_Flag": bool(pd.notna(row["Volume_Ratio_20"]) and row["Volume_Ratio_20"] > 1.0),
                "RS_Flag": bool(pd.notna(row["RS_20D"]) and row["RS_20D"] > 0),
                "Regime_Bull_Flag": bool(row.get("Market_Regime") == "Bull Trend"),
                "ATR_Flag": bool(pd.notna(row["BB_Overshoot_ATR"]) and row["BB_Overshoot_ATR"] <= ATR_OVERSHOOT_MAX),
            }
            # Bandwidth flag needs the PRIOR row (BB width expanding vs. yesterday) — guard pos=0.
            if pos > 0 and pd.notna(row["BB_Width_Pct"]) and pd.notna(feat["BB_Width_Pct"].iloc[pos - 1]):
                event["Bandwidth_Flag"] = bool(row["BB_Width_Pct"] > feat["BB_Width_Pct"].iloc[pos - 1])
            else:
                event["Bandwidth_Flag"] = False

            max_target = entry_pos + MAX_HORIZON - 1
            window_high = high[entry_pos:min(max_target + 1, n)]
            window_low = low[entry_pos:min(max_target + 1, n)]
            if len(window_high) > 0:
                event["MFE_Pct"] = float((window_high.max() - entry_price) / entry_price * 100)
                event["MAE_Pct"] = float((window_low.min() - entry_price) / entry_price * 100)
            else:
                event["MFE_Pct"] = np.nan
                event["MAE_Pct"] = np.nan

            for h in FORWARD_RETURN_HORIZONS:
                target_pos = entry_pos + h - 1
                if target_pos < n:
                    fwd_price = close[target_pos]
                    gross = (fwd_price - entry_price) / entry_price * 100.0
                    event[f"Fwd_Ret_{h}D"] = gross
                    event[f"Fwd_Ret_{h}D_Net"] = gross - ROUND_TRIP_COST_PCT
                else:
                    event[f"Fwd_Ret_{h}D"] = np.nan
                    event[f"Fwd_Ret_{h}D_Net"] = np.nan

            events.append(event)

    return pd.DataFrame(events)


if RUN_RESEARCH_MODE:
    print(f"RESEARCH MODE ENABLED — {RESEARCH_MODE_LABEL}")
    print(f"Entry method: {ENTRY_PRICE_METHOD} | Horizons: {FORWARD_RETURN_HORIZONS} | "
          f"Round-trip cost assumption: {ROUND_TRIP_COST_PCT:.3f}%")
    candidate_events_df = extract_historical_events(stock_feature_frames)
    print(f"Candidate breakout occurrences (Close > Upper BB, any magnitude): {len(candidate_events_df)}")

    if len(candidate_events_df) > 0:
        events_df = candidate_events_df[
            (candidate_events_df["BB_Overshoot_Pct"] > 0) &
            (candidate_events_df["BB_Overshoot_Pct"] <= MAX_BB_OVERSHOOT_PCT) &
            (candidate_events_df["HA_Body_Pct"] >= MIN_HA_BODY_PCT)
        ].reset_index(drop=True)
    else:
        events_df = candidate_events_df.copy()   # still empty — nothing to filter
    print(f"Historical occurrences of the exact MANDATORY setup (overshoot<= {MAX_BB_OVERSHOOT_PCT}%, "
          f"HA body>= {MIN_HA_BODY_PCT}%): {len(events_df)}")
else:
    print("RUN_RESEARCH_MODE is False — skipping historical event study (live scan only).")
    candidate_events_df = pd.DataFrame()
    events_df = pd.DataFrame()

events_df.head(10) if len(events_df) else events_df

RUN_RESEARCH_MODE is False — skipping historical event study (live scan only).


""


## Performance statistics, breakdowns, walk-forward split, A/B table

All numbers below describe what happened **historically in this current-universe simulation** —
they are not a promise about the future, and "sequential equity curve" here means a simple
illustrative compounding of one signal after another in chronological order (it does **not**
model position sizing, capital allocation, or overlapping concurrent trades) — a real backtest
engine would need that layer.

In [31]:
# ============================================================
# SECTION 21 — PERFORMANCE STATISTICS (illustrative — not a portfolio backtest)
# ============================================================
def compute_performance_stats(returns: pd.Series) -> dict:
    r = returns.dropna()
    if len(r) == 0:
        return {
            "Number_of_Signals": 0, "Win_Rate_Pct": np.nan, "Mean_Return_Pct": np.nan,
            "Median_Return_Pct": np.nan, "StdDev_Pct": np.nan, "Max_Drawdown_Pct": np.nan,
            "Profit_Factor": np.nan, "Expectancy_Pct": np.nan, "Avg_Win_Pct": np.nan,
            "Avg_Loss_Pct": np.nan, "Best_Trade_Pct": np.nan, "Worst_Trade_Pct": np.nan,
        }

    wins = r[r > 0]
    losses = r[r <= 0]

    equity_curve = (1 + r / 100.0).cumprod()
    running_max = equity_curve.cummax()
    drawdown = (equity_curve - running_max) / running_max * 100.0

    profit_factor = (wins.sum() / abs(losses.sum())) if losses.sum() != 0 else np.nan

    return {
        "Number_of_Signals": int(len(r)),
        "Win_Rate_Pct": round(len(wins) / len(r) * 100.0, 2),
        "Mean_Return_Pct": round(r.mean(), 3),
        "Median_Return_Pct": round(r.median(), 3),
        "StdDev_Pct": round(r.std(), 3),
        "Max_Drawdown_Pct": round(drawdown.min(), 2),
        "Profit_Factor": round(profit_factor, 2) if pd.notna(profit_factor) else np.nan,
        "Expectancy_Pct": round(r.mean(), 3),
        "Avg_Win_Pct": round(wins.mean(), 3) if len(wins) else np.nan,
        "Avg_Loss_Pct": round(losses.mean(), 3) if len(losses) else np.nan,
        "Best_Trade_Pct": round(r.max(), 3),
        "Worst_Trade_Pct": round(r.min(), 3),
    }


def breakdown_stats(df: pd.DataFrame, group_col: str, horizon: int, net=False) -> pd.DataFrame:
    ret_col = f"Fwd_Ret_{horizon}D" + ("_Net" if net else "")
    rows = []
    for grp_val, sub in df.groupby(group_col, dropna=False):
        stats = compute_performance_stats(sub[ret_col])
        stats[group_col] = grp_val
        rows.append(stats)
    out = pd.DataFrame(rows)
    cols = [group_col] + [c for c in out.columns if c != group_col]
    return out[cols].sort_values("Number_of_Signals", ascending=False)


research_summary_rows = []
forward_returns_detail_df = pd.DataFrame()
breakdown_frames = {}
walk_forward_df = pd.DataFrame()
ab_research_df = pd.DataFrame()

if RUN_RESEARCH_MODE and len(events_df) > 0:
    print(f"\n{RESEARCH_MODE_LABEL}")
    print(f"Total historical signal occurrences: {len(events_df)}\n")

    print("Overall performance by horizon (gross, before estimated transaction costs):")
    for h in FORWARD_RETURN_HORIZONS:
        stats = compute_performance_stats(events_df[f"Fwd_Ret_{h}D"])
        stats["Horizon_Days"] = h
        stats["Cost_Basis"] = "Gross"
        research_summary_rows.append(stats)
        stats_net = compute_performance_stats(events_df[f"Fwd_Ret_{h}D_Net"])
        stats_net["Horizon_Days"] = h
        stats_net["Cost_Basis"] = f"Net (round-trip cost ~{ROUND_TRIP_COST_PCT:.2f}%)"
        research_summary_rows.append(stats_net)

    research_summary_df = pd.DataFrame(research_summary_rows)
    cols = ["Horizon_Days", "Cost_Basis"] + [c for c in research_summary_df.columns if c not in ("Horizon_Days", "Cost_Basis")]
    research_summary_df = research_summary_df[cols]
    print(research_summary_df.to_string(index=False))

    # --- Breakdowns (Part 19) ---
    events_df["HA_Body_Bucket"] = pd.cut(
        events_df["HA_Body_Pct"], bins=[1.0, 1.5, 2.5, 4.0, np.inf],
        labels=["1.0-1.5%", "1.5-2.5%", "2.5-4.0%", ">4.0%"], right=False,
    )
    events_df["ATR_Bucket"] = pd.cut(
        events_df["BB_Overshoot_ATR"], bins=[-np.inf, 0.5, 1.0, 2.0, np.inf],
        labels=["<0.5x ATR", "0.5-1.0x ATR", "1.0-2.0x ATR", ">2.0x ATR"],
    )
    events_df["Volume_Bucket"] = pd.cut(
        events_df["Volume_Ratio_20"], bins=[-np.inf, 0.75, 1.0, 1.5, np.inf],
        labels=["<0.75x avg", "0.75-1.0x avg", "1.0-1.5x avg", ">1.5x avg"],
    )

    for gcol in ["Breakout_Type", "Extension_Class", "Market_Regime", "HA_Body_Bucket", "ATR_Bucket", "Volume_Bucket"]:
        breakdown_frames[gcol] = breakdown_stats(events_df, gcol, PRIMARY_RESEARCH_HORIZON)

    forward_returns_detail_df = events_df.copy()
else:
    research_summary_df = pd.DataFrame([{
        "Note": "RUN_RESEARCH_MODE was False, or no historical signal occurrences were found "
                "in the current universe's available history for this run.",
    }])

In [32]:
# ============================================================
# SECTION 22 — WALK-FORWARD SPLIT (descriptive only) + A/B RESEARCH TABLE
# ============================================================
if RUN_RESEARCH_MODE and len(events_df) > 0:
    train_mask = (events_df["Entry_Date"] >= TRAIN_START) & (events_df["Entry_Date"] <= TRAIN_END)
    test_mask = (events_df["Entry_Date"] >= TEST_START) & (events_df["Entry_Date"] <= TEST_END)

    wf_rows = []
    for label, mask in [("Train (in-sample)", train_mask), ("Test (out-of-sample)", test_mask)]:
        sub = events_df[mask]
        stats = compute_performance_stats(sub[f"Fwd_Ret_{PRIMARY_RESEARCH_HORIZON}D"])
        stats["Period"] = label
        stats["Date_Range"] = (f"{TRAIN_START} to {TRAIN_END}" if "Train" in label
                                 else f"{TEST_START} to {TEST_END}")
        wf_rows.append(stats)
    walk_forward_df = pd.DataFrame(wf_rows)
    walk_forward_df = walk_forward_df[["Period", "Date_Range"] + [c for c in walk_forward_df.columns if c not in ("Period", "Date_Range")]]
    print("Walk-forward split (descriptive only — no threshold was tuned on either period):")
    print(walk_forward_df.to_string(index=False))
    print("\nNOTE: thresholds (MAX_BB_OVERSHOOT_PCT, MIN_HA_BODY_PCT, etc.) were NOT selected using "
          "this Train/Test split or any historical optimization. Comparing Train vs. Test here is "
          "purely to see whether performance is stable across time, not to pick a 'best' period.")

    # --- A/B research table (Part 31) ---
    AB_VERSIONS = {
        "Base": None,
        "Base + Trend": "Trend_Flag",
        "Base + Volume": "Volume_Flag",
        "Base + Bandwidth": "Bandwidth_Flag",
        "Base + ATR": "ATR_Flag",
        "Base + Relative Strength": "RS_Flag",
        "Base + Market Regime": "Regime_Bull_Flag",
    }
    ab_rows = []
    for version_name, flag_col in AB_VERSIONS.items():
        sub = events_df if flag_col is None else events_df[events_df[flag_col] == True]
        row = {"Strategy_Version": version_name, "Signals": len(sub)}
        for h in (5, 10, 20):
            if h not in FORWARD_RETURN_HORIZONS:
                continue
            s = compute_performance_stats(sub[f"Fwd_Ret_{h}D"])
            row[f"Win_Rate_{h}D_Pct"] = s["Win_Rate_Pct"]
            row[f"Median_Return_{h}D_Pct"] = s["Median_Return_Pct"]
            row[f"Mean_Return_{h}D_Pct"] = s["Mean_Return_Pct"]
        row["Max_Adverse_Excursion_Pct"] = round(sub["MAE_Pct"].mean(), 2) if len(sub) else np.nan
        row["Max_Favorable_Excursion_Pct"] = round(sub["MFE_Pct"].mean(), 2) if len(sub) else np.nan
        ab_rows.append(row)
    ab_research_df = pd.DataFrame(ab_rows)
    print("\nA/B research table (Base strategy vs. each optional confirmation, in isolation):")
    print(ab_research_df.to_string(index=False))

    print("\n" + "=" * 70)
    print(f"REMINDER: {RESEARCH_MODE_LABEL}")
    print("This uses TODAY's NIFTY 200 constituents projected back across their own history —")
    print("stocks removed from the index (often after underperforming) are absent by construction.")
    print("Multiple comparisons above (7 versions x 3 horizons, plus 6 sensitivity dimensions later)")
    print("also raise a data-snooping risk: seeing one favorable-looking row is not evidence of an")
    print("edge on its own. Results here do not include real slippage/liquidity constraints beyond")
    print("the flat cost assumption in Configuration, and are not a promise of future performance.")
    print("=" * 70)
else:
    walk_forward_df = pd.DataFrame([{"Note": "Research mode disabled or no historical events found."}])
    ab_research_df = pd.DataFrame([{"Note": "Research mode disabled or no historical events found."}])

## Parameter sensitivity (descriptive only — never auto-optimized)

Two independent grids, exactly as scoped:
1. `MAX_BB_OVERSHOOT_PCT` x `MIN_HA_BODY_PCT` — signal counts always; **mean/median forward return
   and win rate too, when `RUN_RESEARCH_MODE=True`** (reusing the candidate event set above, so no
   extra downloads or recomputation of forward returns).
2. `BB_PERIOD` x `BB_STD_MULT` — signal counts only. Testing these two parameters properly needs
   recomputing the Bollinger engine itself for every combination across full history; that's done
   here directly on the cached raw Close series (still no new downloads), but forward-return
   attribution for this grid is intentionally left out of scope for this notebook to keep the
   runtime bounded — flagged here explicitly rather than silently doing a partial job.

In [33]:
# ============================================================
# SECTION 23 — SENSITIVITY ANALYSIS (descriptive; no auto-optimization)
# ============================================================
# --- Grid 1: MAX_BB_OVERSHOOT_PCT x MIN_HA_BODY_PCT (signal counts + optional forward returns) ---
sens_rows = []
_base_pool = candidate_events_df if (RUN_RESEARCH_MODE and len(candidate_events_df) > 0) else None
_live_pool = ok_df  # today's live scan, at the DEFAULT BB_PERIOD/BB_STD_MULT

for max_overshoot in SENS_OVERSHOOT_GRID:
    for min_ha in SENS_HA_BODY_GRID:
        live_count = int((
            (_live_pool["BB_Overshoot_Pct"] > 0) &
            (_live_pool["BB_Overshoot_Pct"] <= max_overshoot) &
            (_live_pool["HA_Body_Pct"] >= min_ha)
        ).sum())

        row = {"MAX_BB_OVERSHOOT_PCT": max_overshoot, "MIN_HA_BODY_PCT": min_ha,
               "Live_Signals_Today": live_count}

        if _base_pool is not None:
            subset = _base_pool[
                (_base_pool["BB_Overshoot_Pct"] > 0) &
                (_base_pool["BB_Overshoot_Pct"] <= max_overshoot) &
                (_base_pool["HA_Body_Pct"] >= min_ha)
            ]
            row["Historical_Signals"] = len(subset)
            stats = compute_performance_stats(subset[f"Fwd_Ret_{PRIMARY_RESEARCH_HORIZON}D"]) if len(subset) else {}
            row[f"Mean_Ret_{PRIMARY_RESEARCH_HORIZON}D_Pct"] = stats.get("Mean_Return_Pct", np.nan)
            row[f"Median_Ret_{PRIMARY_RESEARCH_HORIZON}D_Pct"] = stats.get("Median_Return_Pct", np.nan)
            row[f"Win_Rate_{PRIMARY_RESEARCH_HORIZON}D_Pct"] = stats.get("Win_Rate_Pct", np.nan)

        sens_rows.append(row)

sensitivity_overshoot_ha_df = pd.DataFrame(sens_rows)
print("Sensitivity grid: MAX_BB_OVERSHOOT_PCT x MIN_HA_BODY_PCT")
print(sensitivity_overshoot_ha_df.to_string(index=False))

# --- Grid 2: BB_PERIOD x BB_STD_MULT (live signal counts only; recomputed on cached Close series) ---
if RUN_RESEARCH_MODE and len(stock_feature_frames) > 0:
    bb_grid_rows = []
    for period in SENS_BB_PERIOD_GRID:
        for std_mult in SENS_BB_STD_MULT_GRID:
            count = 0
            for stock, feat in stock_feature_frames.items():
                close = feat["Close"]
                mid = close.rolling(window=period, min_periods=period).mean()
                sd = close.rolling(window=period, min_periods=period).std(ddof=BB_DDOF)
                upper = mid + std_mult * sd
                last_close, last_upper = close.iloc[-1], upper.iloc[-1]
                if pd.notna(last_upper) and last_close > last_upper:
                    overshoot = (last_close - last_upper) / last_upper * 100
                    ha_body = feat["HA_Body_Pct"].iloc[-1]
                    if 0 < overshoot <= MAX_BB_OVERSHOOT_PCT and pd.notna(ha_body) and ha_body >= MIN_HA_BODY_PCT:
                        count += 1
            bb_grid_rows.append({"BB_PERIOD": period, "BB_STD_MULT": std_mult, "Live_Signals_Today": count})

    sensitivity_bb_params_df = pd.DataFrame(bb_grid_rows)
    print("\nSensitivity grid: BB_PERIOD x BB_STD_MULT (live signal counts only; forward-return "
          "attribution for this grid is out of scope in this notebook — see markdown note above)")
    print(sensitivity_bb_params_df.to_string(index=False))
else:
    sensitivity_bb_params_df = pd.DataFrame([{
        "Note": "Skipped — requires RUN_RESEARCH_MODE=True (needs each stock's full cached "
                "history, which is only retained in memory when research mode is on)."
    }])
    print("\nBB_PERIOD x BB_STD_MULT grid skipped (requires RUN_RESEARCH_MODE=True).")

Sensitivity grid: MAX_BB_OVERSHOOT_PCT x MIN_HA_BODY_PCT
 MAX_BB_OVERSHOOT_PCT  MIN_HA_BODY_PCT  Live_Signals_Today
                  2.0              0.5                   2
                  2.0              1.0                   2
                  2.0              1.5                   1
                  2.0              2.0                   1
                  3.0              0.5                   2
                  3.0              1.0                   2
                  3.0              1.5                   1
                  3.0              2.0                   1
                  4.0              0.5                   3
                  4.0              1.0                   3
                  4.0              1.5                   2
                  4.0              2.0                   2
                  5.0              0.5                   3
                  5.0              1.0                   3
                  5.0              1.5                   2

## Excel export — percentage-format bug fixed (Part 29)

**The bug:** v1 stored `2.31` (meaning 2.31%) and applied Excel's `0.00%` number format, which
tells Excel the underlying value is already a *fraction* — so it rendered `2.31` as `231.00%`.

**The fix (Option A, as requested):** every percentage-point column is divided by 100 immediately
before being written to the sheet (so the stored value becomes `0.0231`), and `0.00%` is applied
to those cells — Excel then displays `2.31%` correctly. This conversion happens **only** at this
export boundary; all internal calculations and threshold comparisons elsewhere in the notebook are
untouched and still work in "percentage points" (matching `MAX_BB_OVERSHOOT_PCT = 4.0`, etc.).

In [34]:
# ============================================================
# SECTION 24 — EXCEL EXPORT (10 sheets; percentage-format bug fixed)
# ============================================================
HEADER_FILL = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
HEADER_FONT = Font(color="FFFFFF", bold=True)
LABEL_FONT = Font(bold=True, italic=True, color="1F4E78")

PRICE_COLUMNS = {
    "CMP", "Close", "BB_Middle", "BB_Upper", "BB_Lower", "HA_Open", "HA_Close",
    "SMA20", "SMA50", "SMA200", "ATR14", "Entry_Price",
    "Index_Close", "Index_SMA20", "Index_SMA50", "Index_SMA200",
}
RATIO_COLUMNS = {
    "Volume_Ratio_20", "Volume_Ratio_50", "BB_PctB", "BB_Overshoot_ATR",
    "Research_Heuristic_Score", "Trend_Score", "Momentum_Score", "Volatility_Score",
    "Candle_Quality_Score", "Liquidity_Score", "Market_Regime_Score", "Profit_Factor",
    "Close_Location_Value", "Dollar_Volume_Percentile",
}
INT_COLUMNS = {
    "Rank", "Days_Above_Upper_BB", "Number_of_Signals", "Signals", "rows_downloaded",
    "rows_valid", "stale_age_days", "Horizon_Days",
}
BIG_NUMBER_COLUMNS = {"Volume", "Avg_Volume_20", "Avg_Volume_50", "Dollar_Volume"}


def _is_pct_point_column(name: str) -> bool:
    """
    True for any column whose numeric value is a PERCENTAGE POINT (e.g. 2.31 meaning 2.31%) that
    still needs the Part-29 fix applied at export time. Centralizing this check in one place is
    what prevents the v1 bug (fixing the format but not the value, or vice versa) from recurring.
    """
    if name in {"RS_20D", "RS_60D", "Stock_Return_20D", "Stock_Return_60D",
                "Index_Return_20D", "Index_Return_60D"}:
        return True
    if "Slope" in name:
        return True
    if name.startswith("Fwd_Ret_"):
        return True
    if "Pct" in name:
        return True
    return False


def prepare_for_excel(df: pd.DataFrame) -> pd.DataFrame:
    """Divides every percentage-point column by 100 so it is a true fraction before Excel applies
    a 0.00% number format to it. This is the ONLY place this conversion happens."""
    out = df.copy()
    for col in out.columns:
        if _is_pct_point_column(col) and pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col] / 100.0
    return out


def _format_block(ws, df: pd.DataFrame, header_row: int, ncols_offset: int = 0):
    max_row = header_row + len(df)
    for col_idx0, col_name in enumerate(df.columns):
        col_idx = col_idx0 + 1 + ncols_offset
        col_letter = get_column_letter(col_idx)
        header_cell = ws.cell(row=header_row, column=col_idx)
        header_cell.fill = HEADER_FILL
        header_cell.font = HEADER_FONT
        header_cell.alignment = Alignment(horizontal="center")

        sample_len = df[col_name].apply(lambda v: len(str(v))).max() if len(df) else 0
        width = min(max(int(sample_len) + 3, len(str(col_name)) + 3, 10), 42)
        ws.column_dimensions[col_letter].width = max(ws.column_dimensions[col_letter].width or 0, width)

        if _is_pct_point_column(col_name):
            fmt = "0.00%"
        elif col_name in PRICE_COLUMNS:
            fmt = "#,##0.00"
        elif col_name in INT_COLUMNS:
            fmt = "0"
        elif col_name in RATIO_COLUMNS:
            fmt = "0.00"
        elif col_name in BIG_NUMBER_COLUMNS:
            fmt = "#,##0"
        else:
            fmt = None

        if fmt:
            for r in range(header_row + 1, max_row + 1):
                ws.cell(row=r, column=col_idx).number_format = fmt


def _write_simple_sheet(writer, sheet_name, df):
    df_out = prepare_for_excel(df)
    df_out.to_excel(writer, sheet_name=sheet_name, index=False)
    ws = writer.sheets[sheet_name]
    ws.freeze_panes = "A2"
    if len(df_out.columns) > 0 and len(df_out) > 0:
        ws.auto_filter.ref = f"A1:{get_column_letter(len(df_out.columns))}{len(df_out) + 1}"
    _format_block(ws, df_out, header_row=1)


def _write_multi_block_sheet(writer, sheet_name, blocks):
    """blocks: list of (label_or_None, dataframe). Stacks them vertically with a blank row + bold
    italic label between blocks."""
    row = 1
    ws = None
    for label, df in blocks:
        df_out = prepare_for_excel(df)
        if label:
            if ws is None:
                df_out.to_excel(writer, sheet_name=sheet_name, index=False, startrow=row, startcol=0)
                ws = writer.sheets[sheet_name]
            else:
                df_out.to_excel(writer, sheet_name=sheet_name, index=False, startrow=row, startcol=0)
            ws.cell(row=row, column=1, value=label).font = LABEL_FONT
            header_row = row + 1
        else:
            df_out.to_excel(writer, sheet_name=sheet_name, index=False, startrow=row - 1, startcol=0)
            if ws is None:
                ws = writer.sheets[sheet_name]
            header_row = row
        _format_block(ws, df_out, header_row=header_row)
        row = header_row + len(df_out) + 3
    return ws


def export_to_excel(filename=OUTPUT_FILENAME):
    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        _write_simple_sheet(writer, "Signals", signals_df)
        _write_simple_sheet(writer, "Diagnostics", diagnostics_df)
        _write_simple_sheet(writer, "Scan_Log", scan_log_df)
        _write_simple_sheet(writer, "Universe", universe_audit_df)
        _write_simple_sheet(writer, "Parameters", params_df)

        _write_multi_block_sheet(writer, "Sensitivity", [
            ("Grid 1: MAX_BB_OVERSHOOT_PCT x MIN_HA_BODY_PCT (live signal counts"
             + (" + historical forward returns)" if RUN_RESEARCH_MODE else ")"), sensitivity_overshoot_ha_df),
            ("Grid 2: BB_PERIOD x BB_STD_MULT (live signal counts only)", sensitivity_bb_params_df),
        ])

        _write_simple_sheet(writer, "Market_Regime", market_regime_summary_df)

        research_blocks = [(f"{RESEARCH_MODE_LABEL}" if RUN_RESEARCH_MODE else "Research mode disabled this run",
                             research_summary_df)]
        if RUN_RESEARCH_MODE and len(events_df) > 0:
            research_blocks.append(("Walk-forward split (descriptive only)", walk_forward_df))
            research_blocks.append(("A/B research table (Base vs. each optional confirmation)", ab_research_df))
            for gcol, bdf in breakdown_frames.items():
                research_blocks.append((f"Breakdown by {gcol} (horizon={PRIMARY_RESEARCH_HORIZON}D)", bdf))
        _write_multi_block_sheet(writer, "Research_Summary", research_blocks)

        _write_simple_sheet(writer, "Forward_Returns", forward_returns_detail_df if len(forward_returns_detail_df)
                             else pd.DataFrame([{"Note": "No historical events (research mode disabled or none found)."}]))

        wb = writer.book
        ws_readme = wb.create_sheet("README")
        ws_readme.column_dimensions["A"].width = 115
        for i, line in enumerate(README_TEXT.split("\n"), start=1):
            cell = ws_readme.cell(row=i, column=1, value=line)
            cell.alignment = Alignment(wrap_text=True, vertical="top")

    return filename


params_df = pd.DataFrame([
    {"Parameter": "Run timestamp (IST)", "Value": RUN_TIMESTAMP_IST.strftime("%Y-%m-%d %H:%M:%S %Z")},
    {"Parameter": "Universe source", "Value": universe_source},
    {"Parameter": "Universe retrieval timestamp", "Value": universe_retrieval_ts.strftime("%Y-%m-%d %H:%M:%S %Z")},
    {"Parameter": "Data provider", "Value": "yfinance (Yahoo Finance)"},
    {"Parameter": "History period requested", "Value": HISTORY_PERIOD},
    {"Parameter": "Data interval", "Value": DATA_INTERVAL},
    {"Parameter": "Auto-adjust prices", "Value": AUTO_ADJUST},
    {"Parameter": "BB_PERIOD", "Value": BB_PERIOD},
    {"Parameter": "BB_STD_MULT", "Value": BB_STD_MULT},
    {"Parameter": "BB_DDOF", "Value": BB_DDOF},
    {"Parameter": "MAX_BB_OVERSHOOT_PCT", "Value": MAX_BB_OVERSHOOT_PCT},
    {"Parameter": "MIN_HA_BODY_PCT", "Value": MIN_HA_BODY_PCT},
    {"Parameter": "MIN_ROWS_PRIMARY", "Value": MIN_ROWS_PRIMARY},
    {"Parameter": "USE_TREND_FILTER", "Value": USE_TREND_FILTER},
    {"Parameter": "USE_LONG_TREND_FILTER", "Value": USE_LONG_TREND_FILTER},
    {"Parameter": "USE_VOLUME_FILTER", "Value": USE_VOLUME_FILTER},
    {"Parameter": "USE_BANDWIDTH_FILTER", "Value": USE_BANDWIDTH_FILTER},
    {"Parameter": "USE_ATR_FILTER", "Value": USE_ATR_FILTER},
    {"Parameter": "USE_CANDLE_QUALITY_FILTER", "Value": USE_CANDLE_QUALITY_FILTER},
    {"Parameter": "USE_RELATIVE_STRENGTH_FILTER", "Value": USE_RELATIVE_STRENGTH_FILTER},
    {"Parameter": "USE_MARKET_REGIME_FILTER", "Value": USE_MARKET_REGIME_FILTER},
    {"Parameter": "USE_FRESH_BREAKOUT_ONLY", "Value": USE_FRESH_BREAKOUT_ONLY},
    {"Parameter": "Benchmark index", "Value": f"{INDEX_TICKER} ({INDEX_NAME})"},
    {"Parameter": "RUN_RESEARCH_MODE", "Value": RUN_RESEARCH_MODE},
    {"Parameter": "ENTRY_PRICE_METHOD", "Value": ENTRY_PRICE_METHOD},
    {"Parameter": "FORWARD_RETURN_HORIZONS", "Value": str(FORWARD_RETURN_HORIZONS)},
    {"Parameter": "ROUND_TRIP_COST_PCT (est.)", "Value": round(ROUND_TRIP_COST_PCT, 3)},
    {"Parameter": "TRAIN_START / TRAIN_END", "Value": f"{TRAIN_START} / {TRAIN_END}"},
    {"Parameter": "TEST_START / TEST_END", "Value": f"{TEST_START} / {TEST_END}"},
    {"Parameter": "Total constituents", "Value": n_total_constituents},
    {"Parameter": "Successfully scanned", "Value": n_scanned_ok},
    {"Parameter": "Failed", "Value": n_failed},
    {"Parameter": "Passing signal", "Value": n_passing},
])

README_TEXT = f"""NIFTY 200 BB + HEIKIN-ASHI RESEARCH SCANNER (v2) — METHODOLOGY

MANDATORY CONDITIONS (unchanged from v1, never silently altered):
  1) Close > Upper Bollinger Band
  2) 0 < BB_Overshoot_Pct <= {MAX_BB_OVERSHOOT_PCT}   where BB_Overshoot_Pct=(Close-Upper)/Upper*100
  3) HA_Body_Pct >= {MIN_HA_BODY_PCT}                  where HA_Body_Pct=(HA_Close-HA_Open)/HA_Open*100

UNIVERSE: NIFTY 200 constituents from NSE's official indices archive. If no official URL validates
(count in [{UNIVERSE_MIN_COUNT},{UNIVERSE_MAX_COUNT}], Symbol column present, no duplicates), the
notebook STOPS rather than silently scanning a small fallback list under the NIFTY 200 label. This
run's source: {universe_source}

PRICES: yfinance, auto_adjust={AUTO_ADJUST}, interval={DATA_INTERVAL}, ~{HISTORY_PERIOD} history.
The whole OHLC block is adjusted together (or not at all) — never mixed.

MINIMUM HISTORY (tiered — fixes the v1 bug that rejected valid constituents like GROWW/ICICIAMC/
LENSKART with <220 rows): a stock is scanned if it has >= {MIN_ROWS_PRIMARY} valid sessions (enough
for BB{BB_PERIOD} + ATR14 + buffer). SMA50/SMA200/RS_60D are computed only when there is enough
history and are otherwise reported as "Unavailable — insufficient history" — never used to reject
the stock outright.

BOLLINGER BANDS: Middle=SMA(Close,{BB_PERIOD}), StdDev=rolling std(Close,{BB_PERIOD},ddof={BB_DDOF}),
Upper/Lower = Middle +/- {BB_STD_MULT}*StdDev. %B=(Close-Lower)/(Upper-Lower) is an additional
diagnostic (Bollinger's own normalized location measure), not a replacement for the mandatory rule.

HEIKIN-ASHI: HA_Close=(O+H+L+C)/4; HA_Open recursive, seeded (Open+Close)/2 at t=0. Synthetic/
averaged — never treated as an actual execution price.

LATEST COMPLETED CANDLE: the newest bar is dropped if it is today (IST) and the NSE session
(close 15:30 IST) hasn't finished. Data_Stale=True is flagged (not silently ignored) if the newest
available bar is more than {STALE_DATA_MAX_DAYS} calendar days old.

MANDATORY vs OPTIONAL: only the three conditions above are mandatory. Trend/volume/bandwidth/ATR/
candle-quality/relative-strength/market-regime/fresh-breakout-only are OPTIONAL (all False by
default) and layer on AFTER the mandatory check — enabling one never changes the mandatory result.
See the Parameters sheet for what was enabled this run.

RANKING: Research_Heuristic_Score is a transparent, documented heuristic (Trend/Momentum/
Volatility/Candle-Quality/Liquidity/Market-Regime components) — explicitly NOT a probability, NOT
an expected return, and NOT validated against realized outcomes unless RUN_RESEARCH_MODE was on
for this run. Overshoot magnitude does NOT feed the score (a stock 3.9% above the band is not
assumed better than one 0.8% above it) — see Extension_Class for the magnitude classification only.

RESEARCH MODE (RUN_RESEARCH_MODE={RUN_RESEARCH_MODE} this run): re-scans each stock's own full
history for every historical occurrence of the exact mandatory setup, using entry at
{ENTRY_PRICE_METHOD} the day after the signal (never same-day close, which would be look-ahead),
and reports forward returns at {FORWARD_RETURN_HORIZONS} trading days, MFE/MAE, and breakdowns.
{RESEARCH_MODE_LABEL if RUN_RESEARCH_MODE else ''}
This is NOT an unbiased point-in-time NIFTY 200 backtest: only TODAY's constituents are used, so
stocks removed from the index historically (often after underperforming) are absent by
construction, which tends to bias results optimistic. Transaction costs are a flat illustrative
assumption (~{ROUND_TRIP_COST_PCT:.2f}% round-trip); no optimization of any threshold was performed.

EXCEL PERCENTAGE FIX (v1 bug, now fixed): percentage-point values (e.g. 2.31 meaning 2.31%) are
divided by 100 at export time and formatted 0.00%, so Excel displays 2.31% instead of 231.00%.

DISCLAIMER: This is a technical screening and research tool. A pass is not a guaranteed breakout,
not a guaranteed profit, not a high-probability win claim, and not investment advice. No setup
described here is "perfect" — performance is regime-dependent, subject to survivorship bias in the
research mode as noted above, and subject to data-snooping risk across the many optional filters,
sensitivity combinations, and A/B comparisons this notebook can produce.
"""

exported_path = export_to_excel()
print(f"Excel workbook written: {exported_path}")

Excel workbook written: Nifty200_BB_HA_Research_Scanner.xlsx


In [35]:
# ============================================================
# SECTION 25 — DOWNLOAD (Colab only)
# ============================================================
try:
    from google.colab import files
    files.download(OUTPUT_FILENAME)
except ImportError:
    print(f"Not running in Colab — file saved locally at: {OUTPUT_FILENAME}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
# ============================================================
# SECTION 26 — FINAL SUMMARY
# ============================================================
print("=" * 46)
print("NIFTY 200 BB + HA RESEARCH SCANNER COMPLETE")
print("=" * 46)
print(f"Universe               : {universe_source}")
print(f"Data Date              : {RUN_TIMESTAMP_IST.strftime('%Y-%m-%d')}")
print(f"Constituents           : {n_total_constituents}")
print(f"Successfully Scanned   : {n_scanned_ok}")
print(f"Failed                 : {n_failed}")
print(f"Primary Signals        : {n_passing}")

if RUN_RESEARCH_MODE and len(events_df) > 0:
    print(f"\nResearch mode          : {RESEARCH_MODE_LABEL}")
    print(f"Historical occurrences : {len(events_df)}")
    _s = compute_performance_stats(events_df[f"Fwd_Ret_{PRIMARY_RESEARCH_HORIZON}D"])
    print(f"  {PRIMARY_RESEARCH_HORIZON}D win rate / mean / median return: "
          f"{_s['Win_Rate_Pct']}% / {_s['Mean_Return_Pct']}% / {_s['Median_Return_Pct']}%")
    print("  (Gross, before estimated transaction costs — see Research_Summary sheet for net figures)")

signals_df

NIFTY 200 BB + HA RESEARCH SCANNER COMPLETE
Universe               : https://nsearchives.nseindia.com/content/indices/ind_nifty200list.csv
Data Date              : 2026-09-09
Constituents           : 200
Successfully Scanned   : 200
Failed                 : 0
Primary Signals        : 3


,Rank,Signal_Date,Stock,Company_Name,Sector,CMP,BB_Middle,BB_Upper,BB_Lower,BB_Width_Pct,BB_PctB,BB_Overshoot_Pct,HA_Open,HA_Close,HA_Body_Pct,HA_Upper_Wick_Pct,HA_Lower_Wick_Pct,ATR14,ATR_Pct,BB_Overshoot_ATR,Volume,Volume_Ratio_20,SMA20,SMA50,SMA200,Dist_SMA50_Pct,Dist_SMA200_Pct,RS_20D,RS_60D,Breakout_Type,Days_Above_Upper_BB,Market_Regime,Research_Heuristic_Score,Signal_Reason
0,1,2026-09-07,SOLARINDS,Solar Industries India Ltd.,Chemicals,21950.00,20195.950000,21769.071132,18622.828868,15.578580,1.057506,0.831128,21079.123857,21762.500000,3.241957,2.075513,0.0,550.642857,2.508623,0.328578,227176.0,1.197564,20195.950000,19146.680469,15685.855293,14.641282,39.934990,NaN,NaN,Fresh Breakout,1,Unavailable — no benchmark index data this run,76.61,"Close Rs 21,950.00 is 0.83% above the Upper Bo..."
1,2,2026-09-08,GVT&D,GE Vernova T&D India Ltd.,Capital Goods,4750.00,4299.738940,4605.837917,3993.639964,14.238026,1.235483,3.129986,4338.009974,4697.275024,8.281794,2.206656,0.0,161.240766,3.394542,0.894080,4274212.0,4.155982,4299.738940,4372.861021,3956.672698,8.624536,20.050365,NaN,NaN,Fresh Breakout,1,Unavailable — no benchmark index data this run,72.72,"Close Rs 4,750.00 is 3.13% above the Upper Bol..."
2,3,2026-09-08,COALINDIA,Coal India Ltd.,Oil Gas & Consumable Fuels,420.25,403.736394,420.067376,387.405411,8.089923,1.005591,0.043475,415.065710,419.500000,1.068334,0.987797,0.0,7.829704,1.863106,0.023324,5273145.0,0.527231,403.736394,410.488101,418.634450,2.378120,0.385909,NaN,NaN,Continuation,4,Unavailable — no benchmark index data this run,47.09,Close Rs 420.25 is 0.04% above the Upper Bolli...


---
### Disclaimer & final engineering review

- **Look-ahead:** the last bar is dropped whenever it belongs to an incomplete NSE session; research
  entries are always T+1, never the signal day's own close.
- **Survivorship bias:** explicitly disclosed wherever research mode output appears — this is a
  current-universe simulation, not a point-in-time backtest.
- **No "perfect strategy" claim:** nothing here is described as guaranteed, high-probability, or
  optimal. Ranking is an explicitly labeled heuristic; sensitivity grids are descriptive only and
  never auto-select a "best" combination.
- **Adjusted vs. unadjusted prices:** the whole OHLC block uses one consistent basis
  (`AUTO_ADJUST`), documented in Configuration and the README sheet — never mixed.
- **Excel percentage bug:** fixed at the export boundary only; internal calculations are untouched.
- **Universe integrity:** a partial/unvalidated universe is never silently scanned and labeled
  "NIFTY 200" — the notebook stops instead (see Section 4a).

This is a technical screening and research tool, not investment advice.